# 21 — Diagnose and improve AHR against KataGo

Build a better AHR player through **earlier-mistake diagnosis → fresh teacher-labelled games → short training cycles → matched evaluation**. Economics weighting is off. The Scout proposes moves, the Worker evaluates them, and AHR performs its own search. KataGo supplies offline training labels and evaluation opponents.

**Start:** select the **AHR Go** kernel and click **Run All**. The last launch cell starts detached tmux session `ahr-v2-k21-gpu0`; the notebook stays available for progress and play. The worker waits if GPU 0 is occupied; GPU 1 can continue its other workload. Repeated Run All detects the same live run and keeps the progress controls available.

**Inputs:** the frozen notebook-19 challenger at step 19535, the existing two-million-position anchor bank, notebooks 00–20 tagged definitions, pinned KataGo assets, and completed development matches. **Outputs:** diagnostic reports, fresh labelled buffers, candidate checkpoints, matched screens, conditional fresh confirmation and a separate notebook-21 verified-player pointer. Reports live under `reports/ahr_v2/<run_id>/`; checkpoints under `checkpoints/<run_id>*/`.

**Runtime:** allow roughly **6–18 hours after GPU 0 becomes available**, subject to the measured 16-game pilot. Each of two cycles is capped at six hours of collection and one active hour of training, with at most 1,000 optimizer updates. Diagnostics and matches add time; unchanged candidates skip matches. These caps do not promise a particular amount of optimizer time or a strength improvement.

**Exact resume:** keep `RUN_CONFIG`, data and tagged definitions unchanged, then click **Run All** again. A live session is detected; otherwise verified controller and child checkpoints resume. After a requested stop, set `RESUME_STOPPED_RUN=True`. Source/configuration mismatches or corrupt manifests raise an error. Latest two valid checkpoints and best are retained, with a single writer and checkpoints at least every 100 updates or 120 seconds for training.

Attach: `tmux attach -t ahr-v2-k21-gpu0`. Detach: **Ctrl-b**, then **d**. The worker survives a browser/SSH disconnect while the instance stays on. The workspace is on the persistent boot disk; deletion policy is unverified and off-instance backup remains deferred at your request.

The official test/lockbox stays unopened. Old evaluation games are consumed development diagnostics, never training examples. UNKNOWN remains UNKNOWN. A bounded tactical proof does not establish full-game optimality; teacher labels and learned improvements are empirical estimates.


**Current repaired run:** `ahr-v2-k21-20261016-gpu0-v1`, isolated to physical GPU 0 (teacher and student use logical `cuda:0`). GPU 1 remains available to the FMM workload. Full-schema conversion at the teacher boundary preserves every history field while avoiding equality errors between independently imported Python state classes; altered histories still fail exact replay. Earlier checksum and forced-query pruning repairs remain active. Validation includes all saved live positions from the failed shard and a real collection → optimizer → search/checkpoint recovery smoke. Earlier runs remain preserved. Run All detects or resumes this registered run.

In [ ]:
from pathlib import Path
import os, nbformat, json
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
PROJECT=Path.cwd().resolve()
while not (PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb').exists():
    if PROJECT.parent==PROJECT:raise FileNotFoundError('Open notebook 21 inside the AHR Go workspace.')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
        exec(compile(cell.source,'00:ahr-export','exec'),globals())
for prefix in ('01_','02_','03_','05_','08_','12_','13_','14_','15_','16_','17_','18_','19_','20_'):
    paths=list((PROJECT/GO2_ROOT).glob(prefix+'*.ipynb'))
    if len(paths)!=1:raise RuntimeError('One canonical dependency required: '+prefix)
    go2_import_exports(paths[0],globals())
from IPython.display import display, Markdown, clear_output, HTML
import pandas as pd
import ipywidgets as widgets
torch.set_num_threads(4)
MODE=os.environ.get('AHR_K21_MODE','run')
if MODE not in ('run','validate','status'):raise ValueError('AHR_K21_MODE must be run, validate, or status.')


## What changes in this experiment

1. Diagnose chronological **early decisions**, then compare Scout-only, full AHR search and search with tactical reading disabled at fixed simulation budgets. Report the first sampled costly move while the teacher still considers the game contestable; sampling does not prove it was the very first mistake.
2. Collect up to **320 fresh self-play games × 32 selected positions per cycle**, aiming for 10,000 valid labels per cycle. The first 16 games provide a throughput estimate. Teacher processes stay loaded within fixed query batches and restart at deterministic batch boundaries; committed responses are stored atomically and reused on resume. This amortizes startup cost while keeping recovery checkable.
3. Mix 75% existing anchor positions with 25% fresh positions. Keep the original teacher policy/value/score/ownership targets, disable economics and the newly introduced observed-outcome loss, and use a small learning rate. Stop after three validation checks without improvement; preserve the step-zero model if it remains best.
4. A candidate that improves validation plays the same 40 KataGo-one-visit opening/color pairs as its base. A predeclared gain of at least four wins with zero UNKNOWN advances it provisionally. A later, separate 100-game confirmation must reach **55 wins**, plus the existing GNU Go regression gate, before changing the notebook-21 playing model.

The iterative teacher/student workflow follows [Expert Iteration](https://arxiv.org/abs/1705.08439). The batching interface uses KataGo’s [analysis engine](https://github.com/lightvector/KataGo/blob/master/docs/Analysis_Engine.md). Neither method guarantees improvement for this model or budget.


## Registered configuration and stage contracts

Definition cells are imported only through the `ahr-export` tag. Production work starts at the explicit Run All launcher.

In [ ]:
GO2_K21_BOOK='experiments/ahr_v2/21_diagnose_and_improve_against_katago.ipynb'
GO2_K21_PARENT='ahr-v2-katago-econ-20261011-v1'

class Go2K21Pause(Exception):
    """A stage paused after its own durable checkpoint."""

def go2_k21_sources(project):
    project=Path(project);sources=go2_k20_control_sources(project)
    sources[GO2_K21_BOOK]=go2_code_sha(project/GO2_K21_BOOK)
    return sources

def go2_k21_bootstrap(project):
    return go2_k20_pipeline_bootstrap(project)+"\ngo2_import_exports(PROJECT/"+repr(GO2_K21_BOOK)+",globals())\ntorch.set_num_threads(4)\n"

def go2_k21_config(run_id='ahr-v2-k21-20261016-gpu0-v1'):
    if Path(run_id).name!=run_id or not run_id.startswith('ahr-v2-'):raise ValueError('A local ahr-v2- run ID is required.')
    return {'schema':'ahr-k21-targeted-learning-v1','run_id':run_id,'purpose':'development','seed':2026101221,
        'device':'cuda:0','teacher_gpu':0,'physical_gpu_uuid':'GPU-e1f3240a-eedb-bed2-dcc1-b9014449ca12','cache_id':'go9x9-competitive-v2','cycles':2,
        'base_run_id':'ahr-k19-20261011-challenger','base_which':'best',
        'base_checkpoint_sha256':'67751114381f36c01fb4b71dfa604a50e2262dfcbb1e1e8638e2fdd593241965',
        'base_model_identity':'bcac91f735d0ebcf54a03ff6718d9f8c659d4a85a7c1fd5c5c6083aeb632ba82',
        'teacher_visits':128,'teacher_batch_size':32,'collection_games':320,'rows_per_game':32,
        'label_rows_target':10000,'collection_shard_games':8,'collection_simulations':32,'collection_reading_nodes':64,'collection_max_plies':300,
        'pilot_games':16,'max_collection_hours':6.0,'min_labelled_rows':1024,
        'diagnostic_games':8,'diagnostic_stride':4,'diagnostic_max_positions':128,
        'optimizer_steps':1000,'learning_rate':1e-5,'batch_size':256,'anchor_fraction':0.75,
        'evaluate_every_steps':100,'early_stop_patience':3,'outcome_weight':0.0,
        'screen_games':40,'screen_min_win_gain':4,'confirmation_games':100,'gnu_regression_games':20,
        'ahr_move_seconds':2.0,'max_simulations':512,'checkpoint_steps':100,'checkpoint_seconds':120,
        'resource_poll_seconds':30,'minimum_free_gib':20,'test_consumed':False,
        'promotion_policy':{'min_kata_games':100,'min_kata_wins':55,'min_gnu_games':20,
            'min_gnu_win_rate':0.70,'max_gnu_drop':0.10,'max_unknown':0,'ahr_soft_move_seconds':2.0,'min_opening_plies':8}}

def go2_k21_check_config(config):
    if config.get('schema')!='ahr-k21-targeted-learning-v1':raise ValueError('Unexpected notebook21 configuration schema.')
    rid=config['run_id']
    if Path(rid).name!=rid or not rid.startswith('ahr-v2-'):raise ValueError('Local run ID required.')
    if config.get('purpose') not in ('development','validation_fixture') or config.get('test_consumed') is not False:
        raise ValueError('Only development or disposable fixture work is permitted; lockbox stays unopened.')
    if config['purpose']=='development':
        if config.get('device')!='cuda:0' or int(config.get('teacher_gpu',-1))!=0:
            raise ValueError('The isolated run requires student and teacher on logical GPU0.')
        if not str(config.get('physical_gpu_uuid','')).startswith('GPU-'):
            raise ValueError('Register the reserved physical GPU UUID explicitly.')
    if not 1<=int(config['cycles'])<=4:raise ValueError('Register between one and four bounded learning cycles.')
    if config['outcome_weight']!=0.0:raise ValueError('The main notebook21 run preserves teacher-only Worker targets; register an isolated ablation separately.')
    if any(int(config[k])<2 or int(config[k])%2 for k in ('screen_games','confirmation_games','gnu_regression_games')):
        raise ValueError('Evaluation game counts must be positive even numbers.')
    if config['checkpoint_steps']>250 or config['checkpoint_seconds']>900:raise ValueError('Checkpoint cadence exceeds the workspace agreement.')
    if config['ahr_move_seconds']!=config['promotion_policy']['ahr_soft_move_seconds']:raise ValueError('Promotion and match budgets differ.')
    return config

def go2_k21_status(project,run_id='ahr-v2-k21-20261016-gpu0-v1'):
    path=Path(project)/'reports/ahr_v2'/run_id/'status.json'
    return json.loads(path.read_text()) if path.exists() else {'status':'NOT_STARTED','stage':None,'run_id':run_id}

def go2_k21_stop(project,run_id):
    go2_atomic_bytes(Path(project)/'checkpoints'/run_id/'STOP',b'Checkpointed stop requested.\n')
    return {'status':'STOP_REQUESTED','run_id':run_id}

def go2_k21_check_stop(project,config):
    if (Path(project)/'checkpoints'/config['run_id']/'STOP').exists():raise Go2K21Pause('Stop requested; resume from saved checkpoints.')

def go2_k21_stages(config):
    if config['purpose']=='validation_fixture':return ['fixture_a','fixture_b','fixture_c','fixture_d']
    return ['diagnosis']+[name+'_'+str(i) for i in range(config['cycles']) for name in ('collect','train','screen')]+['confirmation','promotion']

def go2_k21_initial_selection(config):
    return {'kind':'k19','run_id':config['base_run_id'],'which':config['base_which'],
        'checkpoint_sha256':config['base_checkpoint_sha256'],'model_identity':config['base_model_identity']}

def go2_k21_selected(config,results):
    selected=go2_k21_initial_selection(config)
    for i in range(config['cycles']):
        if 'screen_'+str(i) in results:selected=copy.deepcopy(results['screen_'+str(i)]['selected'])
    return selected

def go2_k21_load_selected(project,selected,device='cpu'):
    loader=go2_k19_load_player if selected['kind']=='k19' else go2_k21_load_player
    model,meta=loader(project,selected['run_id'],which=selected['which'],device=device)
    for key in ('checkpoint_sha256','model_identity'):
        if meta[key]!=selected[key]:raise RuntimeError('Selected candidate changed: '+key)
    return model,meta

def go2_k21_select_screen(base_summary,candidate_summary,min_gain=4):
    required=('WIN','UNKNOWN','attempts')
    if any(k not in s for s in (base_summary,candidate_summary) for k in required):raise ValueError('Complete match counts required.')
    if base_summary['attempts']!=candidate_summary['attempts'] or base_summary['attempts']<2:raise ValueError('Matched screening denominators differ.')
    gain=candidate_summary['WIN']-base_summary['WIN']
    advance=base_summary['UNKNOWN']==candidate_summary['UNKNOWN']==0 and gain>=min_gain
    return {'decision':'ADVANCE' if advance else 'KEEP','win_gain':gain,'required_gain':min_gain,
        'scope':'Exploratory matched development screen; fresh confirmation is still required.'}


In [ ]:
def go2_k21_legacy_config(config):
    result=go2_k20_pipeline_config(config['run_id'])
    result.update(arena_device=config['device'],train_device=config['device'],seed=config['seed'],
        ahr_move_seconds=config['ahr_move_seconds'],max_simulations=config['max_simulations'],cache_id=config['cache_id'])
    return result

def go2_k21_family_inventory(project,config,skip_plans=()):
    project=Path(project);train=set();dev=set();consumed=set();provenance=[]
    for cache in ('go9x9-teacher-v1',config['cache_id']):
        path=project/'data/ahr_v2_teacher/prepared'/cache/'manifest.json';record=json.loads(path.read_text())
        train.update(record['families']['train']);dev.update(record['families']['dev'])
        provenance.append({'path':str(path.relative_to(project)),'sha256':go2_file_sha(path)})
    if train&dev:raise RuntimeError('Prepared training/development opening families overlap.')
    for path in sorted((project/'reports/ahr_v2/benchmarks').glob('*/plan.json')):
        if path.parent.name in skip_plans:continue
        record=json.loads(path.read_text());consumed.update(g['family_sha'] for g in record.get('games',[]) if g.get('opening_actions'))
        provenance.append({'path':str(path.relative_to(project)),'sha256':go2_file_sha(path)})
    old=project/'reports/ahr_v2/benchmarks/ahr-v2-benchmark-20261009-v1/report.json'
    if old.exists():consumed.update(g['opening_family'] for g in json.loads(old.read_text())['games'])
    # Collections include earlier runs and this run's shards/aggregate. Never use an evaluation family for training.
    for path in sorted((project/'checkpoints').glob('ahr-v2-*-collection/manifest.json')):
        if 'fixture' in path.parent.name or path.parent.name.startswith('ahr-v2-k20-buffer-'):continue
        state,entry=go2_load_checkpoint(path.parent)
        if state is None or state.get('config',{}).get('is_smoke') or state.get('config',{}).get('purpose')=='validation_fixture':continue
        families=state.get('sampler',{}).get('opening_families',[])
        train.update(families)
        if families:provenance.append({'path':str(path.relative_to(project)),'family_sha256':go2_digest(sorted(families))})
    if train&dev:raise RuntimeError('Collected training families overlap prepared development families.')
    return {'train':sorted(train),'dev':sorted(dev),'consumed':sorted(consumed),'provenance':provenance}

def go2_k21_arena_namespace(project,snapshot=None):
    ns=go2_k18_namespace(project)
    for prefix in ('19_','20_','21_'):go2_import_exports(next((Path(project)/GO2_ROOT).glob(prefix+'*.ipynb')),ns)
    ns['go2_bench_sources']=ns['go2_k21_sources']
    if snapshot is not None:
        frozen=copy.deepcopy(snapshot)
        def exclusions(project,metadata):
            return {'train':set(frozen['train']),'dev':set(frozen['dev']),'consumed':set(frozen['consumed']),
                'cache_identity':{'k21_exclusions_sha256':go2_digest(frozen)},'prior_reviews':frozen['provenance']}
        ns['go2_bench_prepared_exclusions']=exclusions
    return ns

def go2_k21_plan(project,config,label,profiles):
    project=Path(project);planid=config['run_id']+'-'+label
    path=project/'reports/ahr_v2'/config['run_id']/(label+'-exclusions.json')
    if path.exists():
        record=json.loads(path.read_text());snapshot=record['inventory']
        if record['sha256']!=go2_digest(snapshot):raise RuntimeError('Frozen opening exclusion inventory corrupt.')
        if record['sources']!=go2_k21_sources(project):raise RuntimeError('Opening exclusion source identity changed.')
    else:
        snapshot=go2_k21_family_inventory(project,config,skip_plans=[planid])
        go2_atomic_json(path,{'inventory':snapshot,'sha256':go2_digest(snapshot),'sources':go2_k21_sources(project)})
    legacy=go2_k21_legacy_config(config);cfg=go2_k20_arena_config(legacy,label,profiles)
    ns=go2_k21_arena_namespace(project,snapshot)
    champion,metadata=go2_k20_frozen_incumbent(project,legacy,config['device'])
    plan=ns['go2_bench_plan'](project,cfg,champion,metadata)
    if plan['status']!='COMPLETE':raise Go2K21Pause('Opening plan paused; resume its checkpoint.')
    return plan,cfg

def go2_k21_arena(project,config,label,plan,arena_config,model,metadata,max_events=None):
    go2_k21_check_stop(project,config);ns=go2_k21_arena_namespace(project);cfg=copy.deepcopy(arena_config)
    cfg['run_id']=config['run_id']+'-'+label
    factory=lambda profile,seed,size,komi:ns['go2_bench_opponent_factory'](project,profile,seed,size,komi,
        gtp_timeout_seconds=cfg['gtp_timeout_seconds'],config_directory=Path(project)/'reports/ahr_v2'/config['run_id']/'opponent_configs')
    def progress(report):
        if (Path(project)/'checkpoints'/config['run_id']/'STOP').exists():raise ns['_Go2BenchPause']('User requested checkpointed stop.')
    report=ns['go2_bench_arena'](project,cfg['run_id'],cfg,plan,model,metadata,factory,on_progress=progress,max_events=max_events)
    if report['status']!='COMPLETE':raise Go2K21Pause('Arena paused after a valid checkpoint.')
    return {'report':str((Path('reports/ahr_v2/benchmarks')/cfg['run_id']/'report.json')),
        'summary':report['summary'],'model_identity':metadata['model_identity']}

def go2_k21_stage(project,config,name,results):
    project=Path(project);device=config['device'];selected=go2_k21_selected(config,results)
    go2_k21_check_stop(project,config)
    if name=='diagnosis':
        model,meta=go2_k21_load_selected(project,selected,device)
        return go2_k21_diagnose(project,config,model,meta,on_checkpoint=lambda *_:go2_k21_check_stop(project,config))
    if name.startswith(('collect_','train_','screen_')):
        cycle=int(name.rsplit('_',1)[1]);label='cycle-'+str(cycle+1).zfill(2);buffer_id=config['run_id']+'-'+label+'-buffer'
        if name.startswith('collect_'):
            collection=copy.deepcopy(config);collection.update(buffer_id=buffer_id,seed=config['seed']+cycle*100003)
            value=go2_k21_collect(project,collection,selected['run_id'],base_which=selected['which'],
                on_checkpoint=lambda *_:go2_k21_check_stop(project,config))
            saved,entry=go2_load_checkpoint(project/'checkpoints'/(buffer_id+'-collection'))
            if saved is None or saved.get('stage')!='COMPLETE':raise Go2K21Pause('Collection paused; resume with the unchanged registered limits.')
            count=len(saved['labelled_rows'])
            if count<config['min_labelled_rows']:raise RuntimeError('Collection has only '+str(count)+' valid labels; minimum '+str(config['min_labelled_rows'])+'. Inspect collection report before a separately registered experiment.')
            return {'buffer_id':buffer_id,'labelled_rows':count,'checkpoint':entry,'base':selected,
                'opening_families':sorted(saved['sampler']['opening_families'])}
        if name.startswith('train_'):
            data=results['collect_'+str(cycle)]
            cfg=go2_k21_train_config(config['run_id']+'-'+label+'-teacher-only',data['buffer_id'],device=device,outcome_weight=0.0)
            cfg.update(optimizer_steps=config['optimizer_steps'],learning_rate=config['learning_rate'],batch_size=config['batch_size'],
                anchor_fraction=config['anchor_fraction'],evaluate_every_steps=config['evaluate_every_steps'],patience_evaluations=config['early_stop_patience'],
                seed=config['seed']+cycle*100003,cache_id=config['cache_id'])
            report=go2_k21_train(project,cfg,selected['run_id'],base_which=selected['which'],buffer_id=data['buffer_id'],
                on_checkpoint=lambda *_:go2_k21_check_stop(project,config))
            if report['status'] in ('PAUSED','STOPPED'):raise Go2K21Pause('Learner paused after its checkpoint.')
            saved,entry=go2_load_checkpoint(project/'checkpoints'/cfg['run_id'])
            model,meta=go2_k21_load_player(project,cfg['run_id'],which='best',device='cpu')
            return {'run_id':cfg['run_id'],'best':meta,'latest_checkpoint':entry,'metrics':saved['metrics'],
                'global_step':saved.get('optimizer_step',saved['global_step']),'base':selected,'buffer_id':data['buffer_id'],
                'training_report':{key:report.get(key) for key in ('status','active_seconds','stop_reason')}}
        trained=results['train_'+str(cycle)];meta=trained['best']
        if meta.get('optimizer_step',0)<=0 or not meta.get('development_improved',True):
            return {'decision':'KEEP','reason':'NO_DEVELOPMENT_IMPROVEMENT','selected':selected,'base':selected,
                'candidate_run_id':trained['run_id'],'matches_skipped':True}
        candidate={'kind':'k21','run_id':trained['run_id'],'which':'best','checkpoint_sha256':meta['checkpoint_sha256'],'model_identity':meta['model_identity']}
        profiles=[{'id':'katago-v1','kind':'katago','visits':1,'games':config['screen_games']}]
        plan,arena_cfg=go2_k21_plan(project,config,label+'-screen-plan',profiles)
        base_model,base_meta=go2_k21_load_selected(project,selected,device)
        base=go2_k21_arena(project,config,label+'-screen-base',plan,arena_cfg,base_model,base_meta)
        candidate_model,candidate_meta=go2_k21_load_selected(project,candidate,device)
        challenger=go2_k21_arena(project,config,label+'-screen-candidate',plan,arena_cfg,candidate_model,candidate_meta)
        result=go2_k21_select_screen(base['summary'],challenger['summary'],config['screen_min_win_gain'])
        return result|{'selected':candidate if result['decision']=='ADVANCE' else selected,'base':selected,
            'base_matches':base,'candidate_matches':challenger,'plan_sha256':go2_digest(_go2_bench_plan_pin(plan))}
    if name=='confirmation':
        if selected==go2_k21_initial_selection(config):return {'status':'SKIPPED','reason':'No candidate advanced through the matched screen.','selected':selected}
        model,meta=go2_k21_load_selected(project,selected,device)
        profiles=[{'id':'katago-v1','kind':'katago','visits':1,'games':config['confirmation_games']}]
        kata_plan,kata_cfg=go2_k21_plan(project,config,'confirmation-kata-plan',profiles)
        kata=go2_k21_arena(project,config,'confirmation-kata-candidate',kata_plan,kata_cfg,model,meta)
        profiles=[{'id':'gnugo-l10','kind':'gnugo','level':10,'games':config['gnu_regression_games']}]
        gnu_plan,gnu_cfg=go2_k21_plan(project,config,'confirmation-gnu-plan',profiles)
        candidate=go2_k21_arena(project,config,'confirmation-gnu-candidate',gnu_plan,gnu_cfg,model,meta)
        champion,champ_meta=go2_k20_frozen_incumbent(project,go2_k21_legacy_config(config),device)
        incumbent=go2_k21_arena(project,config,'confirmation-gnu-incumbent',gnu_plan,gnu_cfg,champion,champ_meta)
        inventory=go2_k21_family_inventory(project,config,skip_plans=[config['run_id']+'-confirmation-kata-plan',config['run_id']+'-confirmation-gnu-plan'])
        return {'status':'COMPLETE','selected':selected,'kata':kata,'gnu_candidate':candidate,'gnu_incumbent':incumbent,
            'additional_excluded_families':sorted(set(inventory['train']+inventory['dev']+inventory['consumed']))}
    if name=='promotion':
        confirmation=results['confirmation']
        if confirmation['status']!='COMPLETE':return {'decision':'KEEP_CHAMPION','promoted':False,'reason':confirmation['reason']}
        model,meta=go2_k21_load_selected(project,confirmation['selected'],device)
        evidence={'katago_confirmation':confirmation['kata']['report'],'gnu_candidate':confirmation['gnu_candidate']['report'],
            'gnu_champion':confirmation['gnu_incumbent']['report'],'training_cache_id':config['cache_id'],
            'additional_excluded_families':confirmation['additional_excluded_families']}
        return go2_k21_promote(project,config['run_id'],model,meta,evidence,policy=config['promotion_policy'])
    raise ValueError('Unknown notebook21 stage: '+name)


In [ ]:
def go2_k21_run(project,config,max_stages=None,*,_fixture=False):
    project=go2_project(project);config=copy.deepcopy(config);go2_k21_check_config(config)
    if _fixture!=(config['purpose']=='validation_fixture'):raise ValueError('Fixture dispatch is restricted to disposable fixture configurations.')
    if not _fixture:go2_k21_gpu_runtime(config)
    directory=project/'checkpoints'/config['run_id'];folder=project/'reports/ahr_v2'/config['run_id']
    identity={'schema':config['schema'],'sources':go2_k21_sources(project),'config':config}
    cuda=[] if _fixture else _go2_bench_cuda_devices(config['device'])
    with go2_checkpoint_lock(directory):
        state,entry=go2_load_checkpoint(directory,identity)
        if state is None:
            go2_seed(config['seed'])
            state={'schema':GO2_SCHEMA,'identity':identity,'config':config,'global_step':0,'model':{},'optimizer':{},'scheduler':{},'scaler':{},
                'rng':go2_capture_rng(cuda),'sampler':{'stage_index':0},'metrics':{'completed_stages':0},'results':{},
                'replay':{},'distributed':{'kind':'single_writer_stage_controller'}}
            entry=go2_save_checkpoint(directory,state,best=True,resume_cell='21: Run All with unchanged RUN_CONFIG; RESUME_STOPPED_RUN=True after an intentional stop.')
        else:go2_restore_rng(state['rng'])
        stages=go2_k21_stages(config);done=0;status={}
        while state['sampler']['stage_index']<len(stages):
            name=stages[state['sampler']['stage_index']]
            status={'status':'RUNNING','stage':name,'completed_stages':list(state['results']),
                'run_id':config['run_id'],'checkpoint':entry,'updated_at_utc':go2_now()}
            go2_atomic_json(folder/'status.json',status)
            if (directory/'STOP').exists() or (max_stages is not None and done>=max_stages):break
            try:
                if _fixture:
                    result={'python':random.random(),'numpy':np.random.random(4).tolist(),'torch':torch.rand(4).tolist(),
                        'stage_number':state['sampler']['stage_index']}
                else:result=go2_k21_stage(project,config,name,state['results'])
            except Go2K21Pause:break
            except BaseException as error:
                status.update(status='ERROR',error=repr(error),updated_at_utc=go2_now());go2_atomic_json(folder/'status.json',status);raise
            state['results'][name]=result;state['sampler']['stage_index']+=1;state['global_step']+=1;done+=1
            state['metrics']={'completed_stages':state['sampler']['stage_index']};state['rng']=go2_capture_rng(cuda)
            entry=go2_save_checkpoint(directory,state,best=True,resume_cell='21: Run All; exact stage and child checkpoints resume with unchanged configuration.')
            go2_atomic_json(folder/'results.json',{'results':state['results'],'identity':identity,'checkpoint':entry})
        status={'status':'COMPLETE' if state['sampler']['stage_index']==len(stages) else 'PAUSED',
            'stage':None if state['sampler']['stage_index']==len(stages) else stages[state['sampler']['stage_index']],
            'completed_stages':list(state['results']),'run_id':config['run_id'],'checkpoint':entry,'updated_at_utc':go2_now()}
        go2_atomic_json(folder/'status.json',status)
        return state

def go2_k21_physical_gpu(config):
    """Resolve the registered physical GPU without allocating a CUDA context."""
    expected=str(config['physical_gpu_uuid'])
    query=subprocess.run(['nvidia-smi','--query-gpu=index,uuid,name','--format=csv,noheader,nounits'],
        capture_output=True,text=True,timeout=15)
    if query.returncode:raise RuntimeError('Cannot inspect GPU identity: '+query.stderr.strip()[:200])
    rows=[]
    for line in query.stdout.splitlines():
        parts=[part.strip() for part in line.split(',',2)]
        if len(parts)==3 and parts[1]==expected:rows.append({'physical_index':int(parts[0]),'uuid':parts[1],'name':parts[2]})
    if len(rows)!=1:raise RuntimeError('Registered physical GPU UUID is missing or ambiguous: '+expected)
    return rows[0]

def go2_k21_gpu_runtime(config):
    """Check isolation before any project CUDA work; children inherit this UUID mask."""
    if config.get('purpose')=='validation_fixture':return {'status':'CPU_FIXTURE'}
    expected=str(config['physical_gpu_uuid'])
    if os.environ.get('CUDA_VISIBLE_DEVICES')!=expected:
        raise RuntimeError('Start a fresh worker with CUDA_VISIBLE_DEVICES='+expected+' before importing Torch.')
    if config['device']!='cuda:0' or int(config['teacher_gpu'])!=0:
        raise RuntimeError('The single visible GPU must be logical GPU0 for both teacher and student.')
    physical=go2_k21_physical_gpu(config)
    if not torch.cuda.is_available() or torch.cuda.device_count()!=1:
        raise RuntimeError('Isolated worker must see exactly the registered GPU as logical GPU0.')
    return {'status':'ISOLATED','physical_gpu':physical,'visible_devices':expected,'logical_device':'cuda:0',
        'torch_device_name':torch.cuda.get_device_name(0),'teacher_logical_gpu':0}

def go2_k21_resources(project,config):
    free=shutil.disk_usage(project).free;physical=go2_k21_physical_gpu(config)
    result={'free_gib':round(free/1024**3,1),'required_free_gib':config['minimum_free_gib'],
        'physical_gpu':physical,'foreign_gpu_pids':[],'other_gpu_pids':[],'reserved_sessions':[],
        'reserved_sessions_informational_only':True,'available':False}
    query=subprocess.run(['nvidia-smi','--query-compute-apps=gpu_uuid,pid','--format=csv,noheader,nounits'],
        capture_output=True,text=True,timeout=15)
    if query.returncode:raise RuntimeError('Cannot inspect GPU ownership: '+query.stderr.strip()[:200])
    # Include all descendants, such as the fresh-kernel validation subprocesses.
    parents={}
    for proc in Path('/proc').iterdir():
        if not proc.name.isdigit():continue
        try:parents[int(proc.name)]=int((proc/'stat').read_text().rsplit(')',1)[1].split()[1])
        except (FileNotFoundError,ProcessLookupError,PermissionError,ValueError):pass
    own={os.getpid()}
    while True:
        descendants={pid for pid,parent in parents.items() if parent in own}
        if descendants<=own:break
        own.update(descendants)
    foreign=set();other=set()
    for line in query.stdout.splitlines():
        if not line.strip():continue
        parts=[part.strip() for part in line.split(',')]
        if len(parts)!=2 or not parts[1].isdigit():raise RuntimeError('Unexpected GPU process ownership response: '+line[:160])
        gpu_uuid,pid=parts[0],int(parts[1])
        if pid in own:continue
        (foreign if gpu_uuid==physical['uuid'] else other).add(pid)
    result['foreign_gpu_pids']=sorted(foreign);result['other_gpu_pids']=sorted(other)
    tmux=subprocess.run(['tmux','list-sessions','-F','#{session_name}'],capture_output=True,text=True)
    result['reserved_sessions']=[name for name in tmux.stdout.splitlines() if name.startswith('fmm')]
    result['available']=free>=config['minimum_free_gib']*1024**3 and not result['foreign_gpu_pids']
    return result

def go2_k21_preflight(project,config):
    project=go2_project(project);go2_k21_check_config(config)
    if config['purpose']!='development':raise ValueError('Launcher requires a production development configuration.')
    physical=go2_k21_physical_gpu(config)
    if shutil.which('tmux') is None:raise RuntimeError('tmux is required for the long run.')
    if shutil.disk_usage(project).free<config['minimum_free_gib']*1024**3:raise RuntimeError('Insufficient free persistent disk for the registered run.')
    model,meta=go2_k21_load_selected(project,go2_k21_initial_selection(config),'cpu')
    dataset=go2_k18_dataset(project,config['cache_id'],verify=False)
    if dataset['train']['count']<1000:raise RuntimeError('Prepared anchor data is incomplete.')
    assets=go2_k20_gpu_assets(project,allow_download=False)
    receipt=json.loads((project/'reports/ahr_v2/k21_validation_cpu.json').read_text())
    if receipt.get('status')!='PASS' or receipt.get('sources')!=go2_k21_sources(project):raise RuntimeError('Notebook21 CPU validation is missing or stale; run the validation cell.')
    return {'status':'READY','sources':go2_k21_sources(project),'initial_model':meta['model_identity'],
        'anchor_train_rows':dataset['train']['count'],'free_gib':round(shutil.disk_usage(project).free/1024**3,1),
        'physical_gpu':physical,'logical_teacher_and_student':'cuda:0','worker_visibility':config['physical_gpu_uuid'],
        'storage':'Persistent boot disk; instance-deletion policy unverified; off-instance backup deferred by user.'}

def go2_k21_worker(project,config):
    project=Path(project);directory=project/'checkpoints'/config['run_id'];directory.mkdir(parents=True,exist_ok=True)
    # A worker lock spans resource waiting, validation and controller execution.
    with (directory/'.worker.lock').open('a+b') as lock:
        try:fcntl.flock(lock.fileno(),fcntl.LOCK_EX|fcntl.LOCK_NB)
        except BlockingIOError as error:raise RuntimeError('Another notebook21 worker owns this run.') from error
        go2_k21_preflight(project,config)
        folder=project/'reports/ahr_v2'/config['run_id']
        while True:
            if (directory/'STOP').exists():
                go2_atomic_json(folder/'status.json',{'status':'PAUSED','stage':'resource_wait','run_id':config['run_id'],'updated_at_utc':go2_now()});return
            resources=go2_k21_resources(project,config)
            if resources['available']:break
            go2_atomic_json(folder/'status.json',{'status':'WAITING_FOR_RESOURCES','stage':'resource_wait',
                'run_id':config['run_id'],'resources':resources,'updated_at_utc':go2_now()})
            time.sleep(config['resource_poll_seconds'])
        try:
            gpu_runtime=go2_k21_gpu_runtime(config)
            go2_atomic_json(folder/'gpu_runtime.json',gpu_runtime|{'verified_at_utc':go2_now()})
            go2_atomic_json(folder/'status.json',{'status':'VALIDATING','stage':'native_recovery_smoke','run_id':config['run_id'],'updated_at_utc':go2_now()})
            go2_k21_validate(project,require_native=True,config=config)
            go2_k21_check_stop(project,config)
            return go2_k21_run(project,config)
        except Go2K21Pause:
            go2_atomic_json(folder/'status.json',{'status':'PAUSED','stage':'native_recovery_smoke','run_id':config['run_id'],'updated_at_utc':go2_now()})
        except BaseException as error:
            prior=go2_k21_status(project,config['run_id'])
            prior.update(status='ERROR',error=repr(error),updated_at_utc=go2_now());go2_atomic_json(folder/'status.json',prior);raise

def go2_k21_launch(project,config,session='ahr-v2-k21-gpu0',resume_stopped=False):
    project=go2_project(project);config=copy.deepcopy(config);go2_k21_check_config(config)
    folder=project/'reports/ahr_v2'/config['run_id'];directory=project/'checkpoints'/config['run_id'];sources=go2_k21_sources(project)
    saved_config=folder/'config.json';launch_path=folder/'launch.json'
    if saved_config.exists() and json.loads(saved_config.read_text())!=config:raise RuntimeError('Registered configuration changed; restore it or use a new explicit run ID.')
    if launch_path.exists() and json.loads(launch_path.read_text())['sources']!=sources:raise RuntimeError('Registered source definitions changed; exact resume refused.')
    if (directory/'manifest.json').exists() or list(directory.glob('*.pt')):
        go2_load_checkpoint(directory,{'schema':config['schema'],'sources':sources,'config':config})
    if subprocess.run(['tmux','has-session','-t',session],capture_output=True).returncode==0:
        if not launch_path.exists() or json.loads(launch_path.read_text())['session']!=session:raise RuntimeError('tmux session belongs to another run.')
        return go2_k21_status(project,config['run_id'])|{'session':session,'already_running':True}
    if go2_k21_status(project,config['run_id'])['status']=='COMPLETE':return go2_k21_status(project,config['run_id'])
    stop=directory/'STOP'
    if stop.exists() and not resume_stopped:return {'status':'STOP_REQUESTED','session':session,'run_id':config['run_id']}
    preflight=go2_k21_preflight(project,config)
    if stop.exists():stop.unlink()
    go2_atomic_json(saved_config,config)
    worker=nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(go2_k21_bootstrap(project)),
        nbformat.v4.new_code_cell('CONFIG=json.loads((PROJECT/'+repr(str(saved_config.relative_to(project)))+').read_text())\nRESULT=go2_k21_worker(PROJECT,CONFIG)')],
        metadata={'kernelspec':{'name':'ahr-go','display_name':'AHR Go','language':'python'}})
    path=folder/'worker.ipynb';nbformat.write(worker,path)
    # Set visibility in the command itself: tmux server environment may predate this notebook.
    command=shlex.join(['env','CUDA_VISIBLE_DEVICES='+config['physical_gpu_uuid'],'CUDA_DEVICE_ORDER=PCI_BUS_ID',
        str(project/'.venv/bin/python'),'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace',
        '--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=-1',str(path)])+' >> '+shlex.quote(str(folder/'worker.log'))+' 2>&1'
    launch={'status':'STARTED','session':session,'run_id':config['run_id'],'sources':sources,'config_sha256':go2_digest(config),
        'started_at_utc':go2_now(),'attach':'tmux attach -t '+session,'detach':'Ctrl-b then d','preflight':preflight,
        'worker_environment':{'CUDA_VISIBLE_DEVICES':config['physical_gpu_uuid'],'CUDA_DEVICE_ORDER':'PCI_BUS_ID'}}
    go2_atomic_json(launch_path,launch)
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(project),command],check=True)
    return launch


## Teacher batches, fresh data and earlier-mistake diagnosis

Definition cells are imported only through the `ahr-export` tag. Production work starts at the explicit Run All launcher.

In [ ]:
# Preserve the pinned full-history/rules reader while fixing forced-query pruning.
# Native source: KataGo 92ee95c searchhelpers.cpp isAllowedRootMove.
class Go2K21NativeTeacher(Go2K20OneShotTeacher):
    """Fixed-batch native reader; forced legal moves are not pruned as pointless."""
    def query(self,state,visits=None,allowed_actions=None):
        if state.size!=9 or state.terminal:raise ValueError('Teacher requires a live 9x9 state.')
        self.sequence+=1;visits=int(visits or self.visits)
        if visits<2:raise ValueError('Teacher analysis needs at least two visits.')
        player=state.initial_to_play;moves=[]
        for action in state.actions:
            moves.append(['B' if player==1 else 'W',_go2_bench_gtp_vertex(int(action),9)]);player=-player
        initial=[['B' if v==1 else 'W',_go2_bench_gtp_vertex(i,9)] for i,v in enumerate(state.initial_board) if v]
        request={'id':str(self.sequence),'moves':moves,'initialStones':initial,'initialPlayer':'B' if state.initial_to_play==1 else 'W',
            'rules':copy.deepcopy(GO2_BENCH_KATA_RULES),'komi':state.komi,'boardXSize':9,'boardYSize':9,
            'analyzeTurns':[len(moves)],'maxVisits':visits,'includeOwnership':True,'includePolicy':True,'analysisPVLen':8}
        legal=np.asarray(go2_legal_mask(state),dtype=bool)
        if allowed_actions is not None:
            allowed=[int(a) for a in allowed_actions]
            if not allowed or any(a<0 or a>81 or not legal[a] for a in allowed):raise ValueError('Forced analysis includes an illegal action.')
            request['allowMoves']=[{'player':'B' if state.to_play==1 else 'W','moves':[_go2_bench_gtp_vertex(a,9) for a in allowed],'untilDepth':1}]
            # A legal forced move can lie in pass-alive territory after repeated passes.
            request['overrideSettings']={'rootPruneUselessMoves':False}
        deadline=time.monotonic()+self.timeout_seconds;start=time.monotonic()
        try:self.process.stdin.write((json.dumps(request,separators=(',',':'))+'\n').encode());self.process.stdin.flush()
        except (BrokenPipeError,OSError) as exc:raise RuntimeError('Teacher analysis exited.') from exc
        warnings=[]
        while True:
            while b'\n' not in self.buffer:
                remaining=deadline-time.monotonic()
                if remaining<=0 or not self.selector.select(max(0,remaining)):
                    self.close();raise TimeoutError('Teacher analysis UNKNOWN on timeout.')
                chunk=os.read(self.process.stdout.fileno(),65536)
                if not chunk:raise RuntimeError('Teacher analysis closed output: '+self.stderr_path.read_text()[-1000:])
                self.buffer.extend(chunk)
                if len(self.buffer)>16*1024*1024:raise RuntimeError('Teacher response exceeds bound.')
            line,_,tail=self.buffer.partition(b'\n');self.buffer=bytearray(tail)
            response=json.loads(line)
            if response.get('id') not in [None,request['id']]:raise RuntimeError('Teacher response ID mismatch.')
            if 'error' in response:raise RuntimeError('Teacher analysis error: '+response['error'])
            if 'warning' in response:
                warnings.append(response)
                raise RuntimeError('Teacher warning may change rules/config; strict analysis refused: '+str(response))
            if response.get('isDuringSearch',False):continue
            if 'rootInfo' in response:break
        return go2_k21_decode_teacher_response(self,state,request,response,start)

def go2_k21_decode_teacher_response(worker,state,request,response,start):
    legal=np.asarray(go2_legal_mask(state),dtype=bool)
    root=response['rootInfo']
    if root.get('currentPlayer')!=('B' if state.to_play==1 else 'W'):raise RuntimeError('Teacher perspective mismatch.')
    policy=np.zeros(82,dtype=np.float64);move_values={};move_scores={}
    for item in response.get('moveInfos',[]):
        action=_go2_bench_gtp_action(item['move'],9)
        if legal[action]:
            policy[action]+=max(0,float(item.get('edgeVisits',item['visits'])))
            move_values[action]=2*float(item['winrate'])-1;move_scores[action]=float(item['scoreLead'])
    if policy.sum()<=0:raise RuntimeError('Teacher completed without any compatible searched action.')
    if request.get('allowMoves'):
        allowed={_go2_bench_gtp_action(v,9) for row in request['allowMoves'] for v in row['moves']}
        if any(policy[a]>0 for a in range(82) if a not in allowed):
            raise RuntimeError('Teacher ignored the forced-action restriction.')
    policy/=policy.sum();value=2*float(root['winrate'])-1;score=float(root['scoreLead'])
    ownership=np.asarray(response.get('ownership',[]),dtype=np.float32)
    if not math.isfinite(value) or abs(value)>1.000001 or not math.isfinite(score):raise RuntimeError('Teacher emitted nonfinite value/score.')
    if ownership.shape!=(81,) or not np.isfinite(ownership).all() or (np.abs(ownership)>1.00001).any():raise RuntimeError('Teacher ownership missing/invalid.')
    result={'policy':policy.astype(np.float32),'policy_valid':True,'value':float(value),'score':float(np.clip(score/88.5,-1,1)),
        'score_points':score,'score_valid':abs(score)<=88.5,'ownership':ownership.reshape(9,9),
        'ownership_valid':True,'teacher_estimate':True,'move_values':move_values,'move_scores':move_scores,
        'visits':int(root['visits']),'best_action':int(np.argmax(policy)),'query':request,
        'response':response,'query_sha256':go2_digest(request),'response_sha256':go2_digest(response),
        'identity':worker.identity,'elapsed_seconds':time.monotonic()-start,
        'provenance':'KataGo searched estimate under explicit AHR rules, native pass-alive evaluation; no terminal proof'}
    return result

# Notebook 21 teacher and data protocol. All native code is imported from tagged notebooks.
def go2_k21_teacher_config(config):
    return {'visits':int(config.get('teacher_visits',128)),
        'timeout_seconds':float(config.get('teacher_timeout_seconds',120)),
        'seed':int(config.get('seed',2026101221)),
        'gpu_device':int(config.get('teacher_gpu',config.get('teacher_gpu_device',0))),
        'backend':config.get('teacher_backend','cuda'),
        'batch_size':int(config.get('teacher_query_batch_size',config.get('teacher_batch_size',32))),
        'cache_id':config.get('teacher_cache_id',config['run_id']+'-teacher-cache-'+str(int(config.get('seed',2026101221))))}

class Go2K21Teacher:
    """Amortise engine startup across bounded, exactly replayable sequential query batches.

    The engine remains loaded inside each batch. One native analysis/search thread is
    deliberate: asynchronous multi-thread completion changes KataGo's hidden RNG state.
    Every unfinished batch is regenerated from a fresh engine, fixed order and seed.
    Whole-batch atomic commits precede the recoverable per-query cache indexes.
    """
    def __init__(self,project,visits=128,timeout_seconds=120,seed=2026101221,
                 gpu_device=0,backend='cuda',batch_size=32,cache_id='ahr-v2-k21-teacher-cache',_fixture=None):
        self.project=Path(project);self.visits=int(visits);self.timeout_seconds=float(timeout_seconds)
        self.seed=int(seed);self.gpu_device=int(gpu_device);self.batch_size=int(batch_size);self._fixture=_fixture
        if self.visits<2 or not 1<=self.batch_size<=128 or not 0<self.timeout_seconds<=120:
            raise ValueError('Teacher batches require2+ visits,1–128 queries, and at most120 seconds.')
        if Path(cache_id).name!=cache_id or not cache_id.startswith('ahr-v2-'):raise ValueError('Local AHR cache identifier required.')
        self.assets=({'kind':'synthetic_validation_fixture'} if _fixture is not None else
            (go2_k20_gpu_assets(project) if backend=='cuda' else go2_bench_katago_assets(project,allow_download=False)))
        if backend not in ('cpu','cuda'):raise ValueError('Teacher backend must be explicit.')
        self.identity={'schema':'ahr-k21-deterministic-teacher-batch-v1','sources':go2_k21_sources(project),
            'assets':copy.deepcopy(self.assets),'rules':copy.deepcopy(GO2_BENCH_KATA_RULES),
            'settings':copy.deepcopy(GO2_K20_STATELESS_SETTINGS),
            'forced_query_override':{'rootPruneUselessMoves':False},'seed':self.seed,'visits':self.visits,
            'gpu_device':self.gpu_device,'backend':backend,'batch_size':self.batch_size,
            'timeout_seconds':self.timeout_seconds,'fixed_query_order':True,'whole_batch_atomic_commit':True,
            'native_process_reset':'before every batch, including interrupted batch retry',
            'exactness_scope':'same pinned native executable, model, runtime, device and ordered batch; validated before long work'}
        self.directory=self.project/'checkpoints'/cache_id;self.directory.mkdir(parents=True,exist_ok=True)
        self._active=None;self.counters={'queries_requested':0,'queries_cached':0,'queries_computed':0,'batches_computed':0,'active_seconds':0.0}
        with go2_checkpoint_lock(self.directory):
            manifest=self.directory/'identity.json'
            if manifest.exists() and go2_digest(json.loads(manifest.read_text()))!=go2_digest(self.identity):
                raise RuntimeError('Teacher cache source/configuration identity differs; resume refused.')
            if not manifest.exists():go2_atomic_json(manifest,self.identity)
            self._recover_indexes()
    def _request(self,state,visits=None,allowed_actions=None):
        # Independently imported notebooks define distinct dataclass types. Preserve
        # every history field, then apply exact replay in this teacher's namespace.
        state=go2_state_from_dict(go2_state_to_dict(state))
        if state.size!=9 or state.terminal:raise ValueError('Teacher requires a live9×9 position.')
        go2_verify_replay(state);legal=np.asarray(go2_legal_mask(state),dtype=bool)
        allowed=None if allowed_actions is None else sorted(set(int(a) for a in allowed_actions))
        if allowed is not None and (not allowed or any(a<0 or a>81 or not legal[a] for a in allowed)):
            raise ValueError('Teacher forced action is illegal.')
        value={'state':go2_state_to_dict(state),'visits':int(visits or self.visits),'allowed_actions':allowed,
               'teacher_identity_sha256':go2_digest(self.identity)}
        if value['visits']<2:raise ValueError('Teacher visit budget too small.')
        return value,go2_digest(value)
    def _recover_indexes(self):
        for path in sorted((self.directory/'batches').glob('*.json')):
            record=json.loads(path.read_text());payload=record.get('payload')
            if payload is None or go2_digest(payload)!=record.get('payload_sha256'):
                raise RuntimeError('Committed teacher batch is corrupt: '+str(path))
            if payload.get('teacher_identity_sha256')!=go2_digest(self.identity):raise RuntimeError('Teacher batch identity mismatch.')
            if len(payload['queries'])!=len(payload['answers']):raise RuntimeError('Teacher batch row count mismatch.')
            for index,query in enumerate(payload['queries']):
                key=go2_digest(query);cache=self.directory/'queries'/(key+'.json')
                expected={'batch':path.name,'batch_sha256':record['payload_sha256'],'index':index,'query_sha256':key}
                if cache.exists() and json.loads(cache.read_text())!=expected:
                    raise RuntimeError('Teacher query has conflicting committed batch assignments.')
                if not cache.exists():go2_atomic_json(cache,expected)
    def _cached(self,key):
        path=self.directory/'queries'/(key+'.json')
        if not path.exists():return None
        index=json.loads(path.read_text());batch=self.directory/'batches'/index['batch'];record=json.loads(batch.read_text())
        payload=record['payload']
        if go2_digest(payload)!=record['payload_sha256'] or record['payload_sha256']!=index['batch_sha256']:
            raise RuntimeError('Teacher cache checksum mismatch.')
        query=payload['queries'][index['index']]
        if go2_digest(query)!=key or index['query_sha256']!=key:raise RuntimeError('Teacher cache query mismatch.')
        answer=copy.deepcopy(payload['answers'][index['index']])
        for field in ('move_values','move_scores'):
            answer[field]={int(k):v for k,v in answer.get(field,{}).items()}
        for field in ('policy','ownership'):
            answer[field]=np.asarray(answer[field],dtype=np.float32)
        answer['batch_sha256']=record['payload_sha256'];answer['cache_query_sha256']=key
        return answer
    def _execute_batch(self,queries):
        if not queries or len(queries)>self.batch_size:raise ValueError('Invalid fixed teacher batch size.')
        started=time.monotonic();deadline=started+self.timeout_seconds;answers=[];worker=None
        try:
            if self._fixture is None:
                worker=Go2K21NativeTeacher(self.project,visits=self.visits,timeout_seconds=self.timeout_seconds,
                    seed=self.seed,assets=self.assets,env=self.assets.get('env',{}),gpu_device=self.gpu_device)
                self._active=worker
            for query in queries:
                state=go2_state_from_dict(query['state'])
                if self._fixture is None:
                    remaining=deadline-time.monotonic()
                    if remaining<=0:raise TimeoutError('Teacher batch UNKNOWN on bounded deadline; uncommitted batch is replayed on resume.')
                    worker.timeout_seconds=remaining
                    # Call the verified full-history reader directly; retain this one native process for the fixed batch.
                    answer=worker.query(state,visits=query['visits'],allowed_actions=query['allowed_actions'])
                else:answer=self._fixture(state,query['visits'],query['allowed_actions'],len(answers))
                answer=copy.deepcopy(answer)
                # Match the public cache reader's stable scientific types, including fixtures.
                for field in ('policy','ownership'):
                    answer[field]=np.asarray(answer[field],dtype=np.float32)
                for field in ('move_values','move_scores'):
                    answer[field]={int(k):v for k,v in answer.get(field,{}).items()}
                answer['batch_protocol']=copy.deepcopy(self.identity)
                answers.append(answer)
            return answers,time.monotonic()-started
        finally:
            if worker is not None:worker.close()
            self._active=None
    def query_many(self,items,on_commit=None):
        """items are dict(state,visits?,allowed_actions?) in a reproducible fixed order."""
        prepared=[self._request(item['state'],item.get('visits'),item.get('allowed_actions')) for item in items]
        self.counters['queries_requested']+=len(prepared)
        with go2_checkpoint_lock(self.directory):
            missing=[];seen=set()
            for query,key in prepared:
                if key in seen:continue
                seen.add(key)
                if (self.directory/'queries'/(key+'.json')).exists():self.counters['queries_cached']+=1
                else:missing.append(query)
            for start in range(0,len(missing),self.batch_size):
                batch=missing[start:start+self.batch_size];answers,elapsed=self._execute_batch(batch)
                payload={'teacher_identity_sha256':go2_digest(self.identity),'queries':batch,'answers':answers}
                # JSON object keys become strings on disk. Hash this exact persisted
                # representation so multi-move numeric maps survive a round trip.
                payload=json.loads(json.dumps(payload,sort_keys=True,allow_nan=False,default=go2_json_default))
                digest=go2_digest(payload);name=go2_digest({'teacher':go2_digest(self.identity),'queries':batch})+'.json'
                path=self.directory/'batches'/name
                if path.exists():raise RuntimeError('A committed teacher batch was unexpectedly scheduled twice.')
                go2_atomic_json(path,{'payload_sha256':digest,'payload':payload})
                for index,query in enumerate(batch):
                    key=go2_digest(query)
                    go2_atomic_json(self.directory/'queries'/(key+'.json'),{'batch':name,'batch_sha256':digest,'index':index,'query_sha256':key})
                self.counters['queries_computed']+=len(batch);self.counters['batches_computed']+=1;self.counters['active_seconds']+=elapsed
                if on_commit is not None:on_commit({'batch_sha256':digest,'queries':len(batch),'elapsed_seconds':elapsed})
            result=[self._cached(key) for query,key in prepared]
            if any(answer is None for answer in result):raise RuntimeError('Teacher cache commit incomplete.')
            return result
    def query(self,state,visits=None,allowed_actions=None):
        return self.query_many([{'state':state,'visits':visits,'allowed_actions':allowed_actions}])[0]
    def validate_exact(self):
        state=go2_initial_state(9,7.5)
        for action in (40,10,30,50):state=go2_step(state,action)
        queries=[self._request(state,self.visits,None)[0],self._request(state,self.visits,[20])[0]]
        # Regress the exact late-game position that stopped real collection.
        failure=self.project/'reports/ahr_v2/k21_teacher_repair/native_failure.json'
        if self._fixture is None and failure.is_file():
            record=json.loads(failure.read_text())[0];late=go2_state_from_dict(record['state'])
            forced=[_go2_bench_gtp_action(v,9) for r in record['query']['allowMoves'] for v in r['moves']]
            queries.extend([self._request(late,self.visits,forced)[0]])
        first,_=self._execute_batch(queries);second,_=self._execute_batch(queries)
        for a,b in zip(first,second):
            if not go2_nested_equal(go2_k20_teacher_scientific(a),go2_k20_teacher_scientific(b)):
                raise RuntimeError('Native fixed-batch exact teacher recovery failed; long run refused.')
        # Exercise the real commit/read/reopen path as well as native replay.
        # A separate cache keeps validation queries out of the production cohort.
        with Go2K21Teacher(self.project,visits=self.visits,timeout_seconds=self.timeout_seconds,
                seed=self.seed,gpu_device=self.gpu_device,backend=self.identity['backend'],
                batch_size=self.batch_size,cache_id=self.directory.name+'-roundtrip',_fixture=self._fixture) as cache:
            items=[{'state':go2_state_from_dict(q['state']),'visits':q['visits'],'allowed_actions':q['allowed_actions']} for q in queries]
            stored=cache.query_many(items)
        with Go2K21Teacher(self.project,visits=self.visits,timeout_seconds=self.timeout_seconds,
                seed=self.seed,gpu_device=self.gpu_device,backend=self.identity['backend'],
                batch_size=self.batch_size,cache_id=self.directory.name+'-roundtrip',_fixture=self._fixture) as cache:
            replay=cache.query_many(items)
            if cache.counters['queries_computed']!=0:raise AssertionError('Reopened native cache recomputed committed labels.')
        for a,b,c in zip(first,stored,replay):
            if not go2_nested_equal(go2_k20_teacher_scientific(a),go2_k20_teacher_scientific(b)) or not go2_nested_equal(go2_k20_teacher_scientific(b),go2_k20_teacher_scientific(c)):
                raise RuntimeError('Teacher committed JSON cache changed scientific labels.')
        result={'status':'PASS','persisted_cache_roundtrip_exact':True,'teacher_identity_sha256':go2_digest(self.identity),'native':self._fixture is None,
                'batch_queries':len(queries),'replayed_batches':2,'exact_scientific_responses':True,'updated_at_utc':go2_now()}
        go2_atomic_json(self.directory/'exact-resume-validation.json',result);return result
    def require_validation(self):
        path=self.directory/'exact-resume-validation.json'
        if path.exists():
            result=json.loads(path.read_text())
            if result.get('status')=='PASS' and result.get('teacher_identity_sha256')==go2_digest(self.identity):return result
        return self.validate_exact()
    def close(self):
        if self._active is not None:self._active.close();self._active=None
    def __enter__(self):return self
    def __exit__(self,*args):self.close()


In [ ]:
def go2_k21_data_namespace(project):
    namespace={'__name__':'ahr_k21_collection_exports','PROJECT':Path(project)}
    exec(go2_k20_pipeline_bootstrap(project),namespace)
    return namespace

def go2_k21_data_player(project,run_id,which='best',device='cpu'):
    state,entry=go2_load_checkpoint(Path(project)/'checkpoints'/run_id,which=which)
    if state is None:raise FileNotFoundError('Frozen collection actor checkpoint is missing: '+run_id)
    identity=state['identity']
    if '21' in str(identity.get('training_schema','')) or '21' in str(identity.get('schema','')):
        return go2_k21_load_player(project,run_id,which=which,device=device)
    return go2_k19_load_player(project,run_id,which=which,device=device)

def go2_k21_collection_status(project,run_id):
    path=Path(project)/'reports/ahr_v2'/(run_id+'-buffer-collection')/'status.json'
    return json.loads(path.read_text()) if path.exists() else {'status':'NOT_STARTED'}

def go2_k21_collect(project,config,base_run_id,base_which='best',max_events=None,on_checkpoint=None,*,_fixture=None):
    """Generate fresh self-play in small resumable shards; all labels stay out of diagnostic matches."""
    project=Path(project);config=copy.deepcopy(config);buffer_id=config.get('buffer_id',config['run_id']+'-buffer')
    run_id=buffer_id+'-collection';directory=project/'checkpoints'/run_id
    report_directory=project/'reports/ahr_v2'/run_id;report_directory.mkdir(parents=True,exist_ok=True)
    device=config.get('device','cuda:1');games=int(config.get('collection_games',320));shard_games=int(config.get('collection_shard_games',8))
    target=int(config.get('label_rows_target',10000));minimum=int(config.get('minimum_fresh_rows',config.get('min_labelled_rows',1024)));rows_per_game=int(config.get('rows_per_game',32))
    budget=float(config.get('max_collection_hours',6))*3600
    if games<1 or shard_games<1 or target<minimum or minimum<1 or rows_per_game<1 or budget<=0:raise ValueError('Invalid bounded collection schedule.')
    if _fixture is not None:
        if config.get('purpose')!='validation_fixture' or device!='cpu':raise ValueError('Synthetic collection hooks require an explicit CPU fixture.')
        go2_seed(int(config.get('seed',2026101221)));actor=Go2Network(Go2NetConfig(width=8,blocks=1)).to(device)
        metadata={'run_id':'ahr-v2-k21-synthetic-actor','checkpoint_file':'synthetic','checkpoint_sha256':'synthetic',
            'model_identity':go2_model_identity(actor),'network':dataclasses.asdict(actor.config),'is_smoke':True}
    else:actor,metadata=go2_k21_data_player(project,base_run_id,base_which,device=device)
    actor.eval();actor.requires_grad_(False)
    teacher_kwargs=go2_k21_teacher_config(config)
    if _fixture is not None:teacher_kwargs['_fixture']=lambda position,visits,allowed,index:_Go2K20FixtureTeacher().query(position,visits,allowed)
    with Go2K21Teacher(project,**teacher_kwargs) as teacher,go2_checkpoint_lock(directory):
        teacher_validation=teacher.require_validation()
        previous,previous_entry=go2_load_checkpoint(directory)
        prototype=go2_k20_reward_config(run_id=run_id,base_run_id=base_run_id,base_which=base_which,
            cache_id=config.get('cache_id','go9x9-competitive-v2'),device=device,arm='control',buffer_id=buffer_id)
        if _fixture is not None:prototype['purpose']='validation_fixture'
        if previous is None:
            exclusions=({'families':[],'inputs':[],'frozen_inventory':True,'fixture':True} if _fixture is not None else go2_k20_opening_exclusions(project,prototype))
            # Every previous rollout is training/development evidence, including notebook20 buffers.
            for path in ([] if _fixture is not None else sorted((project/'reports/ahr_v2').glob('*-collection/status.json'))):
                record=json.loads(path.read_text());other_run=record.get('run_id',path.parent.name)
                if other_run==run_id or record.get('purpose')=='validation_fixture' or record.get('collection_identity',{}).get('base_checkpoint',{}).get('checkpoint_file')=='synthetic':continue
                manifest=project/'checkpoints'/other_run/'manifest.json'
                if not manifest.exists():continue
                other,_=go2_load_checkpoint(manifest.parent)
                if other is not None:
                    for item in other.get('replay',[]):
                        if item.get('opening_family'):exclusions['families'].append(item['opening_family'])
                    exclusions['inputs'].append({'file':str(manifest.relative_to(project)),'sha256':go2_file_sha(manifest)})
            exclusions['families']=sorted(set(exclusions['families']))
        else:exclusions=copy.deepcopy(previous['identity']['opening_exclusions'])
        identity={'schema':'ahr-k21-fresh-buffer-v1','sources':go2_k21_sources(project),'config':config,
            'base_checkpoint':{key:metadata.get(key) for key in ('run_id','checkpoint_file','checkpoint_sha256','model_identity')},
            'teacher':teacher.identity,'opening_exclusions':exclusions,'purpose':'fresh_training_only'}
        state,entry=go2_load_checkpoint(directory,identity)
        if state is None:
            state={'schema':GO2_SCHEMA,'identity':identity,'model':copy.deepcopy(actor.state_dict()),'optimizer':{},'scheduler':{},'scaler':{},
                'rng':go2_capture_rng(_go2_k20_cpu_devices(device)),'sampler':{'shard_index':0,'games_committed':0,'opening_families':[]},'global_step':0,
                'config':config,'metrics':{},'stage':'COLLECT','replay':[],'labelled_rows':[],'label_failures':[],
                'model_metadata':metadata,'shards':[],'active_seconds':0.0,'child_config':None,
                'teacher_validation':teacher_validation,'distributed':{'kind':'single_writer_sharded_collection'},'data_cursor':{}}
        else:go2_restore_rng(state['rng'])
        if state['stage']=='COMPLETE':return state
        started=time.monotonic();initial_active=state['active_seconds'];initial_step=state['global_step'];stop_reason=None
        def save(status):
            state['global_step']+=1;state['active_seconds']=initial_active+time.monotonic()-started
            state['rng']=go2_capture_rng(_go2_k20_cpu_devices(device));state['data_cursor']=copy.deepcopy(state['sampler'])
            state['metrics']={'games_finished':len(state['replay']),'labelled_rows':len(state['labelled_rows']),
                'unknown_games':sum(item.get('winner') is None for item in state['replay']),
                'label_failures':len(state['label_failures']),'active_seconds':state['active_seconds']}
            entry=go2_save_checkpoint(directory,state,best=state['stage']=='COMPLETE',
                resume_cell='21: Run All resumes the saved collection configuration; no upstream training cells are imported.')
            report={'run_id':run_id,'status':status,'stage':state['stage'],**state['metrics'],
                'target_rows':target,'minimum_rows':minimum,'requested_games':games,'teacher':copy.deepcopy(teacher.counters),
                'completed_shards':len(state['shards']),'sources':go2_k21_sources(project),'updated_at_utc':go2_now(),
                'completion_reason':state.get('completion_reason'),'production_training_updates':0,'purpose':config.get('purpose','development')}
            if len(state['replay'])>=int(config.get('pilot_games',16)):
                rate=len(state['labelled_rows'])/max(state['active_seconds'],1)
                report['pilot']={'games':len(state['replay']),'labelled_rows_per_hour':rate*3600,
                    'estimated_remaining_hours_for_target':max(0,target-len(state['labelled_rows']))/max(rate,1e-12)/3600,
                    'measurement':'Includes self-play, teacher startup, labels and checkpoint I/O; not optimizer time.'}
            go2_atomic_json(report_directory/'status.json',report)
            if on_checkpoint is not None:on_checkpoint(copy.deepcopy(state),copy.deepcopy(entry))
            return entry
        save('RUNNING')
        try:
            while len(state['replay'])<games and len(state['labelled_rows'])<target:
                if (directory/'STOP').exists():stop_reason='STOPPED';break
                elapsed=initial_active+time.monotonic()-started
                if elapsed>=budget:stop_reason='TIME_BUDGET_COMPLETE';break
                if max_events is not None and state['global_step']-initial_step>=int(max_events):stop_reason='PAUSED';break
                shard=state['sampler']['shard_index'];child_id=buffer_id+'-shard-'+str(shard).zfill(4)
                namespace=go2_k21_data_namespace(project)
                namespace['go2_k20_sources']=go2_k21_sources
                namespace['go2_k19_load_player']=lambda project,run_id,which='best',device='cpu':go2_k21_data_player(project,run_id,which,device)
                namespace['_go2_k20_teacher_pin']=lambda project,cfg:copy.deepcopy(teacher.identity)
                frozen_exclusions=copy.deepcopy(exclusions)
                frozen_exclusions['families']=sorted(set(frozen_exclusions['families'])|{item['opening_family'] for item in state['replay']})
                namespace['go2_k20_opening_exclusions']=lambda project,cfg:copy.deepcopy(frozen_exclusions)
                child_directory=project/'checkpoints'/(child_id+'-collection')
                if state['child_config'] is None:
                    child=copy.deepcopy(prototype);child.update(buffer_id=child_id,run_id=child_id+'-collection',
                        seed=int(config.get('seed',2026101221))+shard*100003,games=min(shard_games,games-len(state['replay'])),
                        rows_per_game=rows_per_game,opponent_cycle=['self'],simulations=int(config.get('collection_simulations',32)),
                        reading_nodes=int(config.get('collection_reading_nodes',64)),reading_wall_seconds=None,
                        search_wall_seconds=None,teacher_backend=config.get('teacher_backend','cuda'),
                        teacher_gpu_device=teacher.gpu_device,teacher_visits=teacher.visits,
                        teacher_timeout_seconds=teacher.timeout_seconds,time_budget_hours=max((budget-elapsed)/3600,0.000001),
                        checkpoint_seconds=120,checkpoint_events=250,max_plies=int(config.get('collection_max_plies',300)))
                    state['child_config']=child;save('RUNNING')
                child=copy.deepcopy(state['child_config'])
                if (child_directory/'STOP').exists() and not (directory/'STOP').exists():(child_directory/'STOP').unlink()
                class BorrowedTeacher:
                    def query(self,*args,**kwargs):return teacher.query(*args,**kwargs)
                    def close(self):pass
                def prefetch(project,cfg):
                    pending,_=go2_load_checkpoint(child_directory)
                    if pending is None:raise RuntimeError('Prefetch requires committed replay positions.')
                    items=[]
                    for row in pending['pending_rows']:
                        position=go2_state_from_dict(row['state']);successor=go2_state_from_dict(row['next_state'])
                        items.extend([{'state':position,'visits':teacher.visits},
                            {'state':position,'visits':teacher.visits,'allowed_actions':[int(row['action'])]}])
                        if not successor.terminal:items.append({'state':successor,'visits':teacher.visits})
                    def commit(record):
                        save('LABELLING')
                        if (directory/'STOP').exists():
                            go2_atomic_bytes(child_directory/'STOP',b'stop after committed teacher batch\n')
                            raise namespace['_Go2K20Pause']()
                    teacher.query_many(items,on_commit=commit)
                    return BorrowedTeacher()
                namespace['_go2_k20_teacher']=prefetch
                def child_checkpoint(child_state,child_entry):
                    save('RUNNING')
                    if child_state['global_step']>0 and ((directory/'STOP').exists() or (max_events is not None and state['global_step']-initial_step>=int(max_events))):
                        raise namespace['_Go2K20Pause']()
                child_state=namespace['go2_k20_collect'](project,child,on_checkpoint=child_checkpoint,_fixture=_fixture)
                if child_state['stage']!='COMPLETE':
                    status_path=project/'reports/ahr_v2'/(child_id+'-collection')/'status.json'
                    child_status=json.loads(status_path.read_text())['status'];stop_reason=child_status
                    break
                overlap={item['opening_family'] for item in child_state['replay']} & set(frozen_exclusions['families'])
                if overlap:raise RuntimeError('Fresh collection opening-family leakage; buffer rejected.')
                _,child_entry=go2_load_checkpoint(child_directory)
                offset=len(state['replay'])
                for row in child_state['labelled_rows']:
                    row=copy.deepcopy(row);row['opening_family']=child_state['replay'][row['game_index']]['opening_family']
                    row['game_index']+=offset;row['source_shard']=child_id
                    row['purpose']='fresh_training';state['labelled_rows'].append(row)
                for game in child_state['replay']:
                    game=copy.deepcopy(game);game['game_index']+=offset;game['source_shard']=child_id;state['replay'].append(game)
                state['label_failures'].extend(copy.deepcopy(child_state['label_failures']))
                state['shards'].append({'run_id':child_id+'-collection','checkpoint':child_entry,'identity':child_state['identity'],
                    'games':len(child_state['replay']),'labelled_rows':len(child_state['labelled_rows'])})
                state['sampler']['shard_index']+=1;state['sampler']['games_committed']=len(state['replay']);state['sampler']['opening_families']=[game['opening_family'] for game in state['replay']];state['child_config']=None
                save('RUNNING')
            if stop_reason in ('STOPPED','PAUSED'):
                save(stop_reason);return state
            if len(state['labelled_rows'])<minimum:
                state['completion_reason']=stop_reason or 'INSUFFICIENT_ROWS';save('INSUFFICIENT_ROWS')
                raise RuntimeError('Fresh collection produced fewer than the required'+str(minimum)+' rows; optimizer training refused.')
            state['stage']='COMPLETE';state['completion_reason']=stop_reason or ('TARGET_ROWS_COMPLETE' if len(state['labelled_rows'])>=target else 'GAME_LIMIT_COMPLETE')
            save('COMPLETE');return state
        except BaseException as exc:
            if isinstance(exc,globals().get('Go2K21Pause',type(None))):
                go2_atomic_json(report_directory/'status.json',{'run_id':run_id,'status':'PAUSED','stage':state['stage'],'updated_at_utc':go2_now()})
                raise
            save('ERROR');raise


In [ ]:
def go2_k21_diagnostic_positions(project,config):
    source=Path(config.get('diagnostic_report',Path(project)/'reports/ahr_v2/benchmarks/ahr-v2-katago-econ-20261011-v1-confirmation-candidate-katago/report.json'))
    source=source.resolve();allowed_root=(Path(project)/'reports/ahr_v2/benchmarks').resolve()
    if allowed_root not in source.parents:raise ValueError('Diagnosis accepts consumed benchmark reports inside the project only.')
    reserved={'final','lockbox','test','official_test'}
    if any(part.lower() in reserved or 'lockbox' in part.lower() for part in source.parts):
        raise ValueError('Final/lockbox/test data must remain unopened.')
    report=json.loads(source.read_text())
    if report.get('purpose')!='development' or report.get('config',{}).get('purpose')!='development':
        raise ValueError('Diagnosis requires an explicitly consumed development report.')
    if report.get('status')!='COMPLETE':raise RuntimeError('Diagnosis requires a complete consumed development report.')
    games=[game for game in report['games'] if game.get('profile')=='katago-v1'][:int(config.get('diagnostic_games',8))]
    if not games:raise RuntimeError('No KataGo one-visit diagnostic games found.')
    stride=int(config.get('diagnostic_stride',4));limit=int(config.get('diagnostic_max_positions',128))
    if stride<1 or limit<1:raise ValueError('Positive diagnosis sampling limits required.')
    per_game=[]
    for game in games:
        state=go2_initial_state(9,7.5);rows=[];ahr_turn=0
        for item in game['trace']:
            if int(item['color'])!=state.to_play:raise RuntimeError('Diagnostic trace perspective mismatch.')
            if item.get('ahr_move',False):
                if ahr_turn%stride==0 and state.move_number<=int(config.get('diagnostic_max_ply',120)):
                    rows.append({'game_index':int(game['game_index']),'profile':game['profile'],'ply':state.move_number,
                        'ahr_color':game['ahr_color'],'played_action':int(item['action']),'state':go2_state_to_dict(state),
                        'purpose':'consumed_development_only','training_eligible':False})
                ahr_turn+=1
            state=go2_step(state,int(item['action']))
        if state.board!=go2_state_from_dict(game['state']).board:raise RuntimeError('Diagnostic replay final board mismatch.')
        per_game.append(rows)
    # Round robin avoids spending the entire position allowance on the first long game.
    positions=[]
    for index in range(max(map(len,per_game),default=0)):
        for rows in per_game:
            if index<len(rows):positions.append(rows[index])
            if len(positions)>=limit:break
        if len(positions)>=limit:break
    return positions,{'path':str(source),'sha256':go2_file_sha(source),'games':len(games),
        'sampling':'chronological early turns, evenly spread across games; first costly sampled move is not necessarily first costly move',
        'purpose':'consumed_development_only','training_rows':0}

def go2_k21_diagnose(project,config,model,metadata,max_events=None,on_checkpoint=None):
    project=Path(project);config=copy.deepcopy(config);run_id=config['run_id']+'-diagnosis'
    directory=project/'checkpoints'/run_id;report_path=project/'reports/ahr_v2'/run_id/'report.json'
    positions,source=go2_k21_diagnostic_positions(project,config)
    device=config.get('device','cuda:1');model.eval();simulations=int(config.get('diagnostic_simulations',128))
    identity={'schema':'ahr-k21-early-decision-diagnosis-v1','sources':go2_k21_sources(project),'config':config,
        'model':{k:metadata.get(k) for k in ('run_id','checkpoint_file','checkpoint_sha256','model_identity')},
        'source_report':source,'positions_sha256':go2_digest(positions),'purpose':'consumed_development_only'}
    with go2_checkpoint_lock(directory),Go2K21Teacher(project,**go2_k21_teacher_config(config)) as teacher:
        validation=teacher.require_validation();state,entry=go2_load_checkpoint(directory,identity)
        if state is None:
            state={'schema':GO2_SCHEMA,'identity':identity,'model':copy.deepcopy(model.state_dict()),'optimizer':{},'scheduler':{},'scaler':{},
                'rng':go2_capture_rng(_go2_k20_cpu_devices(device)),'sampler':{'position_index':0},'global_step':0,'config':config,
                'metrics':{},'stage':'DIAGNOSE','rows':[],'data_cursor':{},'distributed':{'kind':'single_writer'},'teacher_validation':validation}
        else:
            model.load_state_dict(state['model']);go2_restore_rng(state['rng'])
        initial=state['global_step']
        def report(status):
            rows=state['rows'];contestable=[row for row in rows if row.get('contestable',False) and row.get('status')=='COMPLETE']
            first=[]
            for game_index in sorted({row['game_index'] for row in rows}):
                costly=sorted([row for row in contestable if row['game_index']==game_index and row['played_estimated_winrate_regret']>=float(config.get('diagnostic_costly_regret',0.10))],key=lambda r:r['ply'])
                if costly:first.append({key:costly[0][key] for key in ('game_index','ply','teacher_win_probability','played_action','played_estimated_winrate_regret')})
            comparison={}
            for name in ('scout','search','no_reading'):
                eligible=[row[name+'_estimated_winrate_regret'] for row in contestable if name+'_estimated_winrate_regret' in row]
                comparison[name]={'positions':len(eligible),'mean_teacher_estimated_winrate_regret':float(np.mean(eligible)) if eligible else None}
            result={'status':status,'run_id':run_id,'source':source,'positions_planned':len(positions),'positions_completed':len(rows),
                'unknown_positions':sum(row.get('status')=='UNKNOWN' for row in rows),'contestable_positions':len(contestable),
                'first_costly_sampled_moves':first,'matched_search_comparison':comparison,'rows':rows,'training_rows':0,
                'purpose':'consumed_development_only','fixed_simulations':simulations,'updated_at_utc':go2_now(),
                'interpretation':'Teacher estimates diagnose sampled choices, not proofs. Equal simulation counts may take different wall times. Scout uses one network evaluation.'}
            go2_atomic_json(report_path,result);return result
        def save(status):
            state['global_step']+=1;state['rng']=go2_capture_rng(_go2_k20_cpu_devices(device));state['data_cursor']=copy.deepcopy(state['sampler'])
            state['metrics']={'positions_completed':len(state['rows']),'training_rows':0}
            entry=go2_save_checkpoint(directory,state,best=state['stage']=='COMPLETE',resume_cell='21: Run All resumes diagnostic position cursor and immutable teacher batches.')
            report(status)
            if on_checkpoint is not None:on_checkpoint(copy.deepcopy(state),copy.deepcopy(entry))
        if state['stage']=='COMPLETE':return report('COMPLETE')
        save('RUNNING')
        try:
            # Fixed ordered root/played cohort: exact batches do not depend on completion timing.
            items=[]
            for item in positions:
                position=go2_state_from_dict(item['state']);items.extend([{'state':position},{'state':position,'allowed_actions':[item['played_action']]}])
            teacher.query_many(items,on_commit=lambda receipt:save('TEACHER_ROOTS'))
            while state['sampler']['position_index']<len(positions):
                if (directory/'STOP').exists():return report('STOPPED')
                if max_events is not None and state['global_step']-initial>=int(max_events):return report('PAUSED')
                index=state['sampler']['position_index'];item=copy.deepcopy(positions[index]);position=go2_state_from_dict(item['state'])
                row={key:value for key,value in item.items() if key!='state'}
                try:
                    best=teacher.query(position);played=teacher.query(position,allowed_actions=[item['played_action']])
                    probability=(float(best['value'])+1)/2
                    row.update(status='COMPLETE',teacher_win_probability=probability,teacher_best_action=best['best_action'],
                        played_estimated_winrate_regret=max(0.,(best['value']-played['value'])/2),
                        contestable=float(config.get('diagnostic_min_win_probability',0.05))<=probability<=0.95,
                        root_query_sha256=best['cache_query_sha256'])
                    if row['contestable']:
                        start=time.monotonic();evaluation=go2_eval_batch(model,[position])[0];scout=int(np.argmax(evaluation['prior']));scout_seconds=time.monotonic()-start
                        choices={'scout':scout};durations={'scout':scout_seconds};visits={'scout':0}
                        for name,reading in [('search',int(config.get('diagnostic_reading_nodes',128))),('no_reading',0)]:
                            settings=Go2SearchConfig(simulations=simulations,batch_size=8,wall_time_seconds=None,
                                seed=int(config.get('seed',2026101221))+index,root_noise_fraction=0.0,
                                reading_total_nodes=reading,reading_wall_time_seconds=None)
                            start=time.monotonic();search=go2_search(model,position,settings,project=project,
                                context={'purpose':'consumed_development','diagnosis':run_id,'position':index,'variant':name})
                            choices[name]=int(search['action']);durations[name]=time.monotonic()-start;visits[name]=search['completed_simulations']
                            if visits[name]!=simulations:raise RuntimeError('Diagnostic fixed search budget was not completed.')
                        answers=teacher.query_many([{'state':position,'allowed_actions':[choices[name]]} for name in ('scout','search','no_reading')])
                        for name,answer in zip(('scout','search','no_reading'),answers):
                            row[name+'_action']=choices[name];row[name+'_seconds']=durations[name];row[name+'_simulations']=visits[name]
                            row[name+'_estimated_winrate_regret']=max(0.,(best['value']-answer['value'])/2)
                            row[name+'_teacher_query_sha256']=answer['cache_query_sha256']
                        row['worker_value']=float(evaluation['value']);row['scout_search_agree']=scout==choices['search']
                except (TimeoutError,Go2OpponentError) as exc:row.update(status='UNKNOWN',error=repr(exc))
                state['rows'].append(row);state['sampler']['position_index']+=1;save('RUNNING')
            state['stage']='COMPLETE';save('COMPLETE');return report('COMPLETE')
        except BaseException as exc:
            if isinstance(exc,globals().get('Go2K21Pause',type(None))):
                go2_atomic_json(report_path,{'run_id':run_id,'status':'PAUSED','stage':state['stage'],'updated_at_utc':go2_now()})
                raise
            save('ERROR');raise


In [ ]:
def go2_k21_data_validate(project,require_native=False,config=None):
    """Disposable CPU protocol tests; optional real native replay runs only after the user's Run All."""
    project=Path(project);config=copy.deepcopy(config or {'run_id':'ahr-v2-k21-native-validation'})
    # Decoder checks retain searched support and reject fabricated fallback targets.
    position=go2_initial_state(9,7.5);worker=type('DecoderFixture',(),{'identity':{'kind':'synthetic_validation'}})()
    request={'id':'1'}
    good={'rootInfo':{'currentPlayer':'B','winrate':0.6,'scoreLead':1.0,'visits':4},
        'moveInfos':[{'move':'C9','visits':3,'edgeVisits':3,'winrate':0.6,'scoreLead':1.0}],
        'ownership':[0.0]*81}
    decoded=go2_k21_decode_teacher_response(worker,position,request,good,time.monotonic())
    if decoded['policy'][2]!=1.0:raise AssertionError('Searched policy decoding changed.')
    for reason,response,query in (
        ('empty search',good|{'moveInfos':[]},request),
        ('wrong perspective',good|{'rootInfo':good['rootInfo']|{'currentPlayer':'W'}},request),
        ('invalid ownership',good|{'ownership':[]},request),
        ('ignored forced action',good,request|{'allowMoves':[{'moves':['A9'],'player':'B','untilDepth':1}]})):
        try:go2_k21_decode_teacher_response(worker,position,query,response,time.monotonic())
        except RuntimeError:pass
        else:raise AssertionError('Decoder accepted '+reason)

    label='ahr-v2-k21-data-fixture-'+uuid.uuid4().hex[:10]
    def fixture(state,visits,allowed,index):
        legal=np.asarray(go2_legal_mask(state),dtype=bool);policy=legal.astype(float)
        if allowed is not None:
            policy[:]=0
            for action in allowed:policy[action]=1
        policy/=policy.sum();action=int(np.argmax(policy));value=state.to_play*0.25
        request={'id':str(index+1),'state':go2_state_to_dict(state),'visits':visits,'allowed_actions':allowed}
        response={'rootInfo':{'winrate':(value+1)/2,'visits':visits},'moveInfos':[{'move':action,'visits':visits}]}
        return {'policy':policy.astype(np.float32),'policy_valid':True,'value':value,'score':0.0,'score_points':0.0,
            'score_valid':True,'ownership':np.zeros((9,9),dtype=np.float32),'ownership_valid':True,
            'move_values':{2:value,10:value,action:value},'move_scores':{2:0.0,10:0.0,action:0.0},'visits':visits,'best_action':action,
            'query':request,'response':response,'query_sha256':go2_digest(request),'response_sha256':go2_digest(response),
            'elapsed_seconds':0.0,'provenance':'synthetic protocol validation, never production training data'}
    state=go2_initial_state(9,7.5);other=go2_step(state,40)
    items=[{'state':state},{'state':other},{'state':state,'allowed_actions':[20]}]
    with Go2K21Teacher(project,backend='cpu',batch_size=4,cache_id=label,_fixture=fixture) as teacher:
        teacher.validate_exact();first=teacher.query_many(items)
        key=teacher._request(other)[1];index_path=teacher.directory/'queries'/(key+'.json')
        original_index=json.loads(index_path.read_text());index_path.unlink()
    with Go2K21Teacher(project,backend='cpu',batch_size=4,cache_id=label,_fixture=fixture) as resumed:
        if json.loads(index_path.read_text())!=original_index:raise AssertionError('Atomic committed-batch index recovery failed.')
        replay=resumed.query_many(items)
        if not all(go2_nested_equal(go2_k20_teacher_scientific(a),go2_k20_teacher_scientific(b)) for a,b in zip(first,replay)):
            raise AssertionError('Committed teacher response replay changed.')
        if resumed.counters['queries_computed']!=0:raise AssertionError('Resume regenerated a committed label.')
    # A payload mutation must still fail checksum verification after normalization.
    corrupt=label+'-corrupt'
    with Go2K21Teacher(project,backend='cpu',batch_size=4,cache_id=corrupt,_fixture=fixture) as teacher:
        teacher.query_many(items)
        batch_path=next((teacher.directory/'batches').glob('*.json'))
        damaged=json.loads(batch_path.read_text());damaged['payload']['answers'][0]['value']+=0.125
        go2_atomic_json(batch_path,damaged)
        try:teacher.query_many(items)
        except RuntimeError as error:
            if 'checksum mismatch' not in str(error):raise
        else:raise AssertionError('A modified teacher answer passed checksum verification.')
    try:Go2K21Teacher(project,backend='cpu',batch_size=3,cache_id=label,_fixture=fixture)
    except RuntimeError:identity_refused=True
    else:raise AssertionError('Changed teacher batch configuration silently reused the cache.')
    interrupted=label+'-interrupted';calls={'count':0}
    def interrupted_fixture(position,visits,allowed,index):
        calls['count']+=1
        if calls['count']==2:raise TimeoutError('Synthetic interrupted batch remains UNKNOWN.')
        return fixture(position,visits,allowed,index)
    with Go2K21Teacher(project,backend='cpu',batch_size=4,cache_id=interrupted,_fixture=interrupted_fixture) as teacher:
        try:teacher.query_many(items)
        except TimeoutError:pass
        else:raise AssertionError('Interrupted fixture unexpectedly completed.')
        if list((teacher.directory/'batches').glob('*.json')):raise AssertionError('Partial teacher batch was committed.')
    with Go2K21Teacher(project,backend='cpu',batch_size=4,cache_id=interrupted,_fixture=fixture) as teacher:
        recovered=teacher.query_many(items)
        if not all(go2_nested_equal(go2_k20_teacher_scientific(a),go2_k20_teacher_scientific(b)) for a,b in zip(first,recovered)):
            raise AssertionError('Restarted full teacher batch changed scientific labels.')
    positions,source=go2_k21_diagnostic_positions(project,config)
    if not positions or any(row['training_eligible'] or row['purpose']!='consumed_development_only' for row in positions):
        raise AssertionError('Diagnostic rows crossed the training boundary.')
    def collection_fixture(label_suffix):
        return {'run_id':label+'-'+label_suffix,'device':'cpu','teacher_backend':'cpu','purpose':'validation_fixture',
            'seed':2026101221,'collection_games':2,'collection_shard_games':1,'rows_per_game':2,'label_rows_target':4,
            'minimum_fresh_rows':2,'max_collection_hours':0.25,'collection_simulations':2,
            'collection_reading_nodes':0,'collection_max_plies':32,'teacher_visits':4,'teacher_query_batch_size':4}
    full=go2_k21_collect(project,collection_fixture('full'),'ahr-v2-fixture-base',_fixture={})
    partial=go2_k21_collect(project,collection_fixture('resumed'),'ahr-v2-fixture-base',max_events=4,_fixture={})
    if partial['stage']=='COMPLETE':raise AssertionError('Controlled collection interruption did not pause.')
    resumed_collection=go2_k21_collect(project,collection_fixture('resumed'),'ahr-v2-fixture-base',_fixture={})
    def collection_scientific(saved):
        rows=copy.deepcopy(saved['labelled_rows']);replay=copy.deepcopy(saved['replay'])
        for row in rows:row.pop('source_shard',None)
        for game in replay:game.pop('source_shard',None)
        return {'model':saved['model'],'rng':saved['rng'],'sampler':saved['sampler'],
            'rows':rows,'replay':replay,'stage':saved['stage']}
    if not go2_nested_equal(collection_scientific(full),collection_scientific(resumed_collection)):
        first=collection_scientific(full);second=collection_scientific(resumed_collection)
        raise AssertionError('Sharded collection exact recovery differs: '+str([key for key in first if not go2_nested_equal(first[key],second[key])]))
    if any(not row.get('opening_family') for row in full['labelled_rows']):raise AssertionError('Merged rows lost opening-family provenance.')
    native=None
    if require_native:
        with Go2K21Teacher(project,**go2_k21_teacher_config(config)) as teacher:native=teacher.require_validation()
    result={'status':'PASS','sources':go2_k21_sources(project),'exact_scientific_state':True,
        'scope':'teacher batch commit/restart/cache protocol and diagnostic replay; synthetic CPU fixtures contain no optimizer work',
        'searched_response_and_forced_action_checks':True,'empty_response_never_fabricates_policy':True,
        'multi_move_json_checksum_roundtrip_exact':True,'mutated_teacher_payload_rejected':True,
        'committed_batch_missing_index_recovered':True,'two_shard_collection_exact_resume':True,'interrupted_batch_replayed_exactly':True,
        'changed_configuration_refused':identity_refused,'unknown_timeout_preserved':True,
        'diagnostic_positions_verified':len(positions),'diagnostic_training_rows':0,'native':native,
        'native_actual_prediction_exact':bool(native and native.get('exact_scientific_responses')),
        'fixture_run_id':label,'updated_at_utc':go2_now()}
    go2_atomic_json(project/'reports/ahr_v2/k21_validation/data-validation.json',result);return result


## Controlled learning and exact recovery

Definition cells are imported only through the `ahr-export` tag. Production work starts at the explicit Run All launcher.

In [ ]:
GO2_K21_LEARNER_SCHEMA='ahr-k21-controlled-teacher-learning-v1'

class _Go2K21LearnerPause(Exception):
    """A caller may raise this only from an already-committed checkpoint callback."""

def go2_k21_train_config(run_id='ahr-k21-cycle01-teacher',buffer_id='ahr-k21-cycle01-buffer',device='cuda:1',outcome_weight=0.0):
    return {'schema':GO2_K21_LEARNER_SCHEMA,'run_id':run_id,'buffer_id':buffer_id,
        'base_run_id':None,'base_which':'best','cache_id':'go9x9-competitive-v2','device':device,
        'seed':26101221,'batch_size':256,'anchor_fraction':0.75,'learning_rate':1e-5,'weight_decay':1e-4,
        'gradient_clip':2.0,'optimizer_steps':1000,'minimum_lr_fraction':0.1,'precision':'bfloat16' if str(device).startswith('cuda') else 'float32',
        'cpu_threads':4,'checkpoint_seconds':120.0,'checkpoint_optimizer_steps':100,
        'evaluate_every_steps':100,'development_rows':4096,'patience_evaluations':3,'minimum_improvement':1e-4,
        'time_budget_hours':1.0,'is_smoke':False,'fixture_data':False,
        'arm':'teacher_targets' if outcome_weight==0 else 'outcome_ablation',
        'row_weight_scope':'policy','loss_weights':{'policy':1.0,'teacher_value':1.0,'game_outcome':float(outcome_weight),
            'score':0.5,'ownership':0.25,'economic_policy':0.0,'economic_value':0.0},
        'selection_basis':'fixed_teacher_only_development_loss_not_playing_strength',
        'target_contract':'teacher_policy_value_score_ownership_preserved; observed_outcome_separate; economics_off'}

def go2_k21_validate_train_config(config):
    default=go2_k21_train_config();unknown=set(config)-set(default)
    if unknown:raise ValueError('Unknown notebook21 learner settings: '+str(sorted(unknown)))
    result={**default,**copy.deepcopy(config)}
    for key in ('schema','row_weight_scope','selection_basis','target_contract'):
        if result[key]!=default[key]:raise ValueError('Changed learner contract: '+key)
    for key in ('run_id','buffer_id','cache_id'):
        if not isinstance(result[key],str) or not result[key] or Path(result[key]).name!=result[key]:raise ValueError('Simple local ID required: '+key)
    if result['base_run_id'] is not None and (not result['base_run_id'] or Path(result['base_run_id']).name!=result['base_run_id']):raise ValueError('Invalid base run ID')
    if result['base_which'] not in ('best','latest'):raise ValueError('Choose best or latest base')
    if result['device'] not in ('cpu','cuda:0','cuda:1'):raise ValueError('Use CPU or explicit GPU')
    for key in ('seed','batch_size','cpu_threads','optimizer_steps','checkpoint_optimizer_steps','evaluate_every_steps','development_rows','patience_evaluations'):
        if type(result[key]) is not int or result[key]<1:raise ValueError('Positive integer required: '+key)
    if result['batch_size']%4:raise ValueError('Batch size must divide into exact 75/25 anchor and fresh rows')
    for key in ('learning_rate','gradient_clip','checkpoint_seconds','time_budget_hours'):
        if not math.isfinite(result[key]) or result[key]<=0:raise ValueError('Positive finite setting required: '+key)
    if not math.isfinite(result['minimum_improvement']) or result['minimum_improvement']<0:raise ValueError('Invalid minimum improvement')
    if result['anchor_fraction']!=.75:raise ValueError('This registered comparison uses exactly 75% anchor rows')
    if result['checkpoint_seconds']>120 or result['checkpoint_optimizer_steps']>100:raise ValueError('Checkpoint cadence must be <=120 seconds and <=100 optimizer steps')
    if not 0<result['minimum_lr_fraction']<=1 or not math.isfinite(result['weight_decay']) or result['weight_decay']<0:raise ValueError('Invalid optimizer settings')
    if result['precision'] not in ('float32','bfloat16') or (result['device']=='cpu' and result['precision']!='float32'):raise ValueError('Invalid precision/device')
    if result['fixture_data'] and not result['is_smoke']:raise ValueError('Synthetic fixture data must be marked as smoke')
    if not result['is_smoke'] and (result['optimizer_steps']>2000 or result['evaluate_every_steps']>100):raise ValueError('Use short measured cycles (<=2000 steps, development every <=100)')
    weights=result['loss_weights'];outcome=weights.get('game_outcome')
    expected={**default['loss_weights'],'game_outcome':outcome}
    if outcome not in (0.,.5) or weights!=expected:raise ValueError('Only the separately registered observed-outcome weight may vary (0 or 0.5)')
    if result['arm']!=('teacher_targets' if outcome==0 else 'outcome_ablation'):raise ValueError('Arm name and outcome objective disagree')
    return result

def go2_k21_base_player(project,run_id,which='best',device='cpu'):
    state,_=go2_load_checkpoint(Path(project)/'checkpoints'/run_id,which=which)
    if state is None:raise RuntimeError('Registered pretrained base checkpoint missing')
    if state['identity'].get('training_schema')==GO2_K21_LEARNER_SCHEMA:
        return go2_k21_load_player(project,run_id,which=which,device=device)
    if state['identity'].get('training_schema')==GO2_K19_SCHEMA:
        return go2_k19_load_player(project,run_id,which=which,device=device)
    raise RuntimeError('Notebook21 accepts an ordinary notebook19 or notebook21 Go2Network base')

def go2_k21_teacher_row_batch(rows,indices,device='cpu'):
    chosen=[rows[int(index)] for index in indices]
    if not chosen:raise ValueError('Nonempty fresh batch required')
    states=[go2_state_from_dict(row['state']) for row in chosen]
    outcomes=[]
    for row in chosen:
        winner=row.get('winner');value=None if winner is None else int(row['colour'])*winner
        outcomes.append([float(value==-1),float(value==0),float(value==1)] if winner is not None else [0.,0.,0.])
    values={'features':np.stack([go2_encode_state(state) for state in states]),
        'legal':np.stack([go2_legal_mask(state) for state in states]),
        'policy':np.asarray([row['teacher_policy'] for row in chosen]),
        'value':np.asarray([row['teacher_value'] for row in chosen]),
        'score':np.asarray([row['teacher_score'] for row in chosen]),
        'score_valid':np.asarray([row['teacher_score_valid'] for row in chosen]),
        'ownership':np.asarray([row['teacher_ownership'] for row in chosen]).reshape(-1,81),
        'ownership_valid':np.asarray([row['teacher_ownership_valid'] for row in chosen]),
        'outcome':np.asarray(outcomes),'outcome_valid':np.asarray([row.get('winner') is not None for row in chosen]),
        'row_weight':np.ones(len(chosen)),'policy_valid':np.ones(len(chosen),dtype=bool),
        'value_valid':np.ones(len(chosen),dtype=bool)}
    flags={'legal','score_valid','ownership_valid','outcome_valid','policy_valid','value_valid'}
    result={key:torch.as_tensor(value,dtype=torch.bool if key in flags else torch.float32,device=device) for key,value in values.items()}
    go2_k19_validate_batch(result)
    return result

def go2_k21_join_teacher_batches(anchor,fresh):
    keys=('features','legal','policy','value','score','score_valid','outcome','outcome_valid','ownership','ownership_valid','row_weight','policy_valid','value_valid')
    prepared=[]
    for original in (anchor,fresh):
        batch=dict(original);n=len(batch['features']);device=batch['features'].device
        batch['ownership']=batch['ownership'].reshape(n,81)
        for key in ('policy_valid','value_valid'):batch.setdefault(key,torch.ones(n,dtype=torch.bool,device=device))
        batch.setdefault('row_weight',torch.ones(n,device=device));prepared.append(batch)
    return {key:torch.cat([batch[key] for batch in prepared]) for key in keys}

def go2_k21_learner_fixture(project,config):
    dataset=go2_k19_fixture_dataset(project);go2_seed(config['seed']+901)
    model=Go2Network(Go2NetConfig(width=8,blocks=1));states=[];position=go2_initial_state(9,7.5)
    for action in (40,30,41,31,50,49,42,32):
        states.append(position);position=go2_step(position,action)
    rows=[]
    for index,position in enumerate(states):
        legal=np.asarray(go2_legal_mask(position));policy=legal.astype(float);policy/=policy.sum()
        rows.append({'state':go2_state_to_dict(position),'colour':position.to_play,'winner':(-1 if index%2 else 1) if index%3 else None,
            'teacher_policy':policy.tolist(),'teacher_value':float(np.sin(index)*.5),'teacher_score':float(np.sin(index)*.1),
            'teacher_score_valid':index%3!=0,'teacher_ownership':np.zeros(81).tolist(),'teacher_ownership_valid':index%2==0,
            'final_score_black':80.,'final_ownership_black':np.ones(81).tolist(),'opening_family':'synthetic-fixture'})
    metadata={'run_id':'k21-disposable-base','checkpoint_file':'synthetic','checkpoint_sha256':go2_tensor_digest(model.state_dict()),
        'model_identity':go2_model_identity(model),'network':dataclasses.asdict(model.config),'is_smoke':True,'optimizer_step':0}
    buffer={'identity':{'schema':'k21-disposable-buffer-v1','source_pin':'deterministic-synthetic-v1'},'stage':'COMPLETE','labelled_rows':rows,'model_metadata':metadata}
    return dataset,model,metadata,buffer,{'file':'synthetic','sha256':go2_digest(rows)}

def go2_k21_learner_inputs(project,config):
    if config['fixture_data']:return go2_k21_learner_fixture(project,config)
    base,metadata=go2_k21_base_player(project,config['base_run_id'],config['base_which'],device='cpu')
    if metadata.get('is_smoke'):raise RuntimeError('A development learner cannot initialize from synthetic weights')
    buffer,entry=go2_load_checkpoint(project/'checkpoints'/(config['buffer_id']+'-collection'))
    if buffer is None or buffer.get('stage')!='COMPLETE' or not buffer.get('labelled_rows'):raise RuntimeError('Completed immutable teacher buffer required; collection is never run implicitly')
    source=buffer['identity'].get('sources')
    if source is None or go2_digest(source)!=go2_digest(go2_k21_sources(project)):raise RuntimeError('Teacher buffer source identity differs')
    actor=buffer.get('model_metadata',{})
    for key in ('run_id','checkpoint_file','checkpoint_sha256','model_identity'):
        if actor.get(key)!=metadata.get(key):raise RuntimeError('Buffer actor and training initialization differ: '+key)
    dataset=go2_k18_dataset(project,cache_id=config['cache_id'],verify=True)
    return dataset,base,metadata,buffer,entry


In [ ]:
def go2_k21_learner_report(project,state,entry,status):
    metrics=state['metrics']
    report={'schema':GO2_K21_LEARNER_SCHEMA,'run_id':state['config']['run_id'],'status':status,'updated_at_utc':go2_now(),
        'optimizer_step':state['global_step'],'optimizer_steps_requested':state['config']['optimizer_steps'],
        'active_optimizer_seconds':state['active_seconds'],'metrics':copy.deepcopy(metrics),'sampler':copy.deepcopy(state['sampler']),
        'arm':state['config']['arm'],'outcome_weight':state['config']['loss_weights']['game_outcome'],'economics_enabled':False,
        'teacher_worker_target_preserved':True,'teacher_score_ownership_preserved':True,
        'buffer_id':state['config']['buffer_id'],'base_checkpoint':state['identity']['initialization'],
        'checkpoint':entry,'stop_reason':state.get('stop_reason'),'is_smoke':state['config']['is_smoke'],
        'selection_is_playing_strength_evidence':False,'test_consumed':False}
    go2_atomic_json(Path(project)/'reports/ahr_v2'/state['config']['run_id']/'status.json',report)
    return report

def go2_k21_train(project,config,base_run_id=None,base_which='best',buffer_id=None,max_steps=None,on_checkpoint=None):
    """Resume one frozen short cycle; max_steps is an additional interruption allowance, never part of its scientific identity."""
    project=go2_project(project);config=go2_k21_validate_train_config(config)
    if base_run_id is not None:
        if config['base_run_id'] not in (None,base_run_id):raise ValueError('Conflicting base run IDs')
        config['base_run_id']=base_run_id;config['base_which']=base_which
    if not config['base_run_id'] and not config['fixture_data']:raise ValueError('Explicit pretrained base run required')
    if buffer_id is not None and buffer_id!=config['buffer_id']:raise ValueError('Conflicting immutable buffer IDs')
    if max_steps is not None and (type(max_steps) is not int or max_steps<1):raise ValueError('max_steps must be an additional positive update allowance')
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8');torch.set_num_threads(config['cpu_threads'])
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True;torch.use_deterministic_algorithms(True)
    dataset,base,metadata,buffer,buffer_entry=go2_k21_learner_inputs(project,config)
    train_rows=_go2_k20_rows(dataset,'train');dev_rows=_go2_k20_rows(dataset,'dev')
    if min(train_rows,dev_rows)<1:raise RuntimeError('Separate nonempty training and development anchor splits required')
    rows=buffer['labelled_rows'];base_pin={key:metadata.get(key) for key in ('run_id','checkpoint_file','checkpoint_sha256','model_identity','optimizer_step')}
    identity={'schema':GO2_SCHEMA,'training_schema':GO2_K21_LEARNER_SCHEMA,'sources':go2_k21_sources(project),'config':copy.deepcopy(config),
        'initialization':base_pin,'network':dataclasses.asdict(base.config),'model_type':'Go2Network',
        'data':go2_k19_data_identity(dataset),'buffer_identity':copy.deepcopy(buffer['identity']),
        'buffer_checkpoint':{'file':buffer_entry['file'],'sha256':buffer_entry['sha256']},'buffer_rows_sha256':go2_digest(rows),
        'opening_families':sorted({row.get('opening_family','') for row in rows}),
        'selection_contract':'fixed anchor development, teacher-only loss; actual game strength requires independent matches'}
    directory=project/'checkpoints'/config['run_id'];go2_seed(config['seed'])
    model=Go2Network(Go2NetConfig(**identity['network'])).to(config['device']);model.load_state_dict(base.state_dict(),strict=True);model.requires_grad_(True)
    optimizer=torch.optim.AdamW(model.parameters(),lr=config['learning_rate'],weight_decay=config['weight_decay'])
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda step:config['minimum_lr_fraction']+(1-config['minimum_lr_fraction'])*.5*(1+math.cos(math.pi*min(step/config['optimizer_steps'],1.))))
    scaler=torch.amp.GradScaler('cuda',enabled=False);generator=torch.Generator(device='cpu').manual_seed(config['seed']+21)
    selection_config=copy.deepcopy(config);selection_config['loss_weights']['game_outcome']=0.0
    anchor_count=config['batch_size']*3//4;fresh_count=config['batch_size']-anchor_count
    sample=go2_k19_sample_batch
    with go2_checkpoint_lock(directory):
        state,entry=go2_load_checkpoint(directory,identity)
        if state is None:
            state={'schema':GO2_SCHEMA,'identity':identity,'config':config,'model':{},'optimizer':{},'scheduler':{},'scaler':{},'rng':{},
                'sampler':{'rows_seen':0,'anchor_rows_seen':0,'fresh_rows_seen':0,'batches':0},'global_step':0,
                'metrics':{'last_loss':None,'recent_losses':[],'development_history':[],'last_development':None,
                    'baseline_development_loss':None,'best_development_loss':None,'best_optimizer_step':0,'nonimproving_evaluations':0,
                    'observed_outcome_rows':0,'early_stopping_triggered':False},
                'development_indices':torch.randperm(dev_rows,generator=torch.Generator().manual_seed(config['seed']+55))[:min(dev_rows,config['development_rows'])],
                'active_seconds':0.0,'stop_reason':None,'adapters':{},'replay':{'buffer_rows_sha256':identity['buffer_rows_sha256'],'buffer_identity':buffer['identity']},
                'distributed':{'kind':'single_writer_single_learner'},'data_cursor':{'optimizer_step':0},'base_metadata':copy.deepcopy(metadata)}
        else:
            model.load_state_dict(state['model'],strict=True);optimizer.load_state_dict(state['optimizer']);scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
            go2_restore_rng(state['rng'],{'learner':generator});go2_invalidate_model_identity(model)
        last_saved=-1 if entry is None else state['global_step'];last_saved_time=time.monotonic();last_saved_step=state['global_step'];processed=0;pending_best=False
        def save(force=False):
            nonlocal entry,last_saved,last_saved_time,last_saved_step,pending_best
            if state['global_step']==last_saved:return
            if not force and time.monotonic()-last_saved_time<config['checkpoint_seconds'] and state['global_step']-last_saved_step<config['checkpoint_optimizer_steps']:return
            state.update(model=copy.deepcopy(model.state_dict()),optimizer=copy.deepcopy(optimizer.state_dict()),scheduler=copy.deepcopy(scheduler.state_dict()),
                scaler=copy.deepcopy(scaler.state_dict()),rng=go2_capture_rng(_go2_k20_cpu_devices(config['device']),{'learner':generator}))
            state['data_cursor']['optimizer_step']=state['global_step']
            entry=go2_save_checkpoint(directory,state,best=pending_best,
                resume_cell='21: load checkpoints/<run_id>/config.json and call go2_k21_train(PROJECT, saved_config); same source/data/base pins are required')
            pending_best=False;last_saved=state['global_step'];last_saved_step=state['global_step'];last_saved_time=time.monotonic()
            go2_k21_learner_report(project,state,entry,'RUNNING')
            if on_checkpoint is not None:
                saved_rng=go2_capture_rng(_go2_k20_cpu_devices(config['device']),{'learner':generator})
                try:on_checkpoint(copy.deepcopy(state),copy.deepcopy(entry))
                finally:go2_restore_rng(saved_rng,{'learner':generator})
        def evaluate():
            nonlocal pending_best
            audit=go2_k19_metric_evaluate(model,dataset,state['development_indices'],selection_config,sample_fn=sample)
            audit['optimizer_step']=state['global_step'];audit['selection_outcome_weight']=0.0
            metrics=state['metrics'];metrics['last_development']=audit;metrics['development_history'].append(audit)
            value=float(audit['joint_loss']);old=metrics['best_development_loss']
            if old is None:
                metrics['baseline_development_loss']=value;metrics['best_development_loss']=value;metrics['best_optimizer_step']=0;pending_best=True
            elif value<old-config['minimum_improvement']:
                metrics['best_development_loss']=value;metrics['best_optimizer_step']=state['global_step'];metrics['nonimproving_evaluations']=0;pending_best=True
            else:metrics['nonimproving_evaluations']+=1
            if metrics['nonimproving_evaluations']>=config['patience_evaluations']:
                metrics['early_stopping_triggered']=True;state['stop_reason']='DEVELOPMENT_EARLY_STOP'
        go2_atomic_json(directory/'config.json',config)
        status='RUNNING'
        try:
            if state['metrics']['last_development'] is None:evaluate()
            save(True)
            if state['stop_reason'] is not None:status='EARLY_STOPPED' if state['stop_reason']=='DEVELOPMENT_EARLY_STOP' else 'COMPLETE'
            else:
                while state['global_step']<config['optimizer_steps']:
                    if (directory/'STOP').exists():status='STOPPED';break
                    if state['active_seconds']>=3600*config['time_budget_hours']:status='TIME_BUDGET_COMPLETE';break
                    if max_steps is not None and processed>=max_steps:status='PAUSED';break
                    start=time.monotonic();anchor_ids=torch.randint(0,train_rows,(anchor_count,),generator=generator)
                    fresh_ids=torch.randint(0,len(rows),(fresh_count,),generator=generator)
                    anchor=sample(dataset,anchor_ids,device=config['device'],split='train')
                    fresh=go2_k21_teacher_row_batch(rows,fresh_ids,device=config['device'])
                    batch=go2_k21_join_teacher_batches(anchor,fresh)
                    batch=go2_k19_d4(batch,torch.randint(0,8,(config['batch_size'],),generator=generator))
                    model.train();optimizer.zero_grad(set_to_none=True)
                    with torch.autocast(device_type='cuda' if str(config['device']).startswith('cuda') else 'cpu',dtype=torch.bfloat16,enabled=config['precision']=='bfloat16'):
                        result=go2_k19_loss(model,batch,config)
                    result['loss'].backward();torch.nn.utils.clip_grad_norm_(model.parameters(),config['gradient_clip'],error_if_nonfinite=True)
                    optimizer.step();scheduler.step();go2_invalidate_model_identity(model)
                    if str(config['device']).startswith('cuda'):torch.cuda.synchronize(torch.device(config['device']))
                    state['active_seconds']+=time.monotonic()-start;state['global_step']+=1;processed+=1
                    state['sampler']['rows_seen']+=config['batch_size'];state['sampler']['anchor_rows_seen']+=anchor_count
                    state['sampler']['fresh_rows_seen']+=fresh_count;state['sampler']['batches']+=1
                    metrics=state['metrics'];metrics['last_loss']=float(result['loss'].detach().cpu())
                    metrics['observed_outcome_rows']+=int(batch['outcome_valid'].sum())
                    metrics['recent_losses']=(metrics['recent_losses']+[{'optimizer_step':state['global_step'],'loss':metrics['last_loss'],
                        'learning_rate':float(optimizer.param_groups[0]['lr']),'components':{key:float(value.detach().cpu()) for key,value in result['components'].items()}}])[-128:]
                    boundary=state['global_step']%config['evaluate_every_steps']==0 or state['global_step']==config['optimizer_steps']
                    if boundary:evaluate()
                    if state['global_step']==config['optimizer_steps'] and state['stop_reason'] is None:state['stop_reason']='MAX_OPTIMIZER_STEPS'
                    save(boundary)
                    if state['stop_reason'] is not None:
                        status='EARLY_STOPPED' if state['stop_reason']=='DEVELOPMENT_EARLY_STOP' else 'COMPLETE';break
                else:status='COMPLETE'
        except (_Go2K21LearnerPause,globals().get('Go2K21Pause',_Go2K21LearnerPause)):
            status='PAUSED'
        except BaseException as exc:
            previous,previous_entry=go2_load_checkpoint(directory,identity)
            if previous is not None:
                report=go2_k21_learner_report(project,previous,previous_entry,'ERROR');report.update(error=repr(exc),discarded_partial_update=True)
                go2_atomic_json(project/'reports/ahr_v2'/config['run_id']/'status.json',report)
            raise
        save(True);go2_k21_learner_report(project,state,entry,status)
        state['status']=status
        return state

def go2_k21_load_player(project,run_id,which='best',device='cpu'):
    project=go2_project(project);state,entry=go2_load_checkpoint(project/'checkpoints'/run_id,which=which)
    if state is None or state['identity'].get('training_schema')!=GO2_K21_LEARNER_SCHEMA:raise RuntimeError('Verified notebook21 learner checkpoint required')
    if go2_digest(state['identity']['sources'])!=go2_digest(go2_k21_sources(project)):raise RuntimeError('Notebook21 exported source identity changed')
    model=Go2Network(Go2NetConfig(**state['identity']['network'])).to(device);model.load_state_dict(state['model'],strict=True)
    model.eval();model.requires_grad_(False)
    metadata={'run_id':run_id,'which':which,'global_step':state['global_step'],'optimizer_step':state['global_step'],'learner_optimizer_step':state['global_step'],
        'checkpoint_file':entry['file'],'checkpoint_sha256':entry['sha256'],'model_identity':go2_model_identity(model),'model_type':'Go2Network',
        'network':state['identity']['network'],'sources':state['identity']['sources'],'is_smoke':state['config']['is_smoke'],
        'data_identity':{**state['identity']['data'],'buffer':state['identity']['buffer_identity'],'buffer_rows_sha256':state['identity']['buffer_rows_sha256']},'buffer_identity':state['identity']['buffer_identity'],'opening_families':state['identity']['opening_families'],
        'development_metrics':state['metrics']['last_development'],'development_improved':state['metrics']['best_optimizer_step']>0,
        'base_metadata':copy.deepcopy(state['base_metadata']),'arm':state['config']['arm'],'economics_enabled':False,
        'worker_target':'teacher value','teacher_used_to_choose_moves':False,'true_objective':'terminal win'}
    return model,metadata


In [ ]:
def go2_k21_learner_scientific_state(state):
    return {key:copy.deepcopy(state[key]) for key in ('model','optimizer','scheduler','scaler','rng','sampler','global_step',
        'development_indices','metrics','adapters','replay','data_cursor','stop_reason')}

def go2_k21_learner_fixture_worker(project,config,steps,label):
    from nbclient import NotebookClient
    folder=Path(project)/'reports/ahr_v2'/config['run_id'];folder.mkdir(parents=True,exist_ok=True)
    path=folder/(label+'.ipynb')
    worker=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('Disposable CPU learner recovery fixture. Synthetic labels; no strength claim.'),
        nbformat.v4.new_code_cell(go2_k21_bootstrap(project)),
        nbformat.v4.new_code_cell('CONFIG=json.loads('+repr(json.dumps(config))+')\nSTATE=go2_k21_train(PROJECT,CONFIG,max_steps='+repr(steps)+')\nprint({"optimizer_step":STATE["global_step"],"status":STATE["status"]})')],
        metadata={'kernelspec':{'display_name':'AHR Go','language':'python','name':'ahr-go'}})
    nbformat.write(worker,path);NotebookClient(worker,timeout=240,kernel_name='ahr-go',resources={'metadata':{'path':str(project)}}).execute();nbformat.write(worker,path)
    state,_=go2_load_checkpoint(Path(project)/'checkpoints'/config['run_id']);return state

def go2_k21_validate_learner(project=None,device='cpu'):
    project=go2_project(project)
    if device not in ('cpu','cuda:0','cuda:1'):raise ValueError('Explicit validation device required')
    if device!='cpu' and (not torch.cuda.is_available() or torch.cuda.device_count()<=int(device[-1])):raise RuntimeError('Requested validation GPU unavailable')
    token=uuid.uuid4().hex[:10];config=go2_k21_train_config('k21-learner-fixture-'+token+'-continuous','synthetic',device=device)
    config.update(fixture_data=True,is_smoke=True,batch_size=8,optimizer_steps=6,evaluate_every_steps=2,development_rows=16,
        patience_evaluations=100,checkpoint_optimizer_steps=2,minimum_improvement=0.,cpu_threads=1)
    dataset,model,_,buffer,_=go2_k21_learner_fixture(project,config)
    row_batch=go2_k21_teacher_row_batch(buffer['labelled_rows'],[0,1,2,3])
    # Observed results must not silently replace teacher score or ownership.
    expected=torch.tensor([row['teacher_score'] for row in buffer['labelled_rows'][:4]],dtype=torch.float32)
    if not torch.equal(row_batch['score'],expected) or not torch.equal(row_batch['ownership'],torch.zeros(4,81)):raise AssertionError('Observed game result replaced teacher auxiliary targets')
    batch=go2_k19_sample_batch(dataset,np.arange(8),device='cpu');model.eval()
    normal=go2_k19_loss(model,batch,config);changed=copy.deepcopy(batch);changed['outcome']=batch['outcome'].flip(1)
    if not torch.equal(normal['loss'],go2_k19_loss(model,changed,config)['loss']):raise AssertionError('Default Worker objective depends on observed outcomes')
    ablation=copy.deepcopy(config);ablation['loss_weights']['game_outcome']=.5;ablation['arm']='outcome_ablation';go2_k21_validate_train_config(ablation)
    if torch.equal(go2_k19_loss(model,batch,ablation)['loss'],go2_k19_loss(model,changed,ablation)['loss']):raise AssertionError('Registered outcome ablation has no effect')
    transformed=go2_k19_d4(batch,torch.arange(8))
    if not torch.equal(transformed['outcome'],batch['outcome']) or not torch.equal(transformed['value'],batch['value']):raise AssertionError('D4 changed scalar outcomes or teacher value')
    if (transformed['policy'].masked_select(~transformed['legal']).abs()>1e-6).any():raise AssertionError('D4 policy and legal mask are misaligned')
    full=go2_k21_learner_fixture_worker(project,config,6,'continuous')
    partial_cfg={**copy.deepcopy(config),'run_id':'k21-learner-fixture-'+token+'-resumed'}
    interrupted=go2_k21_learner_fixture_worker(project,partial_cfg,3,'interrupted')
    if interrupted['global_step']!=3 or interrupted['sampler']['rows_seen']!=24:raise AssertionError('Partial learner cursor was not saved')
    resumed=go2_k21_learner_fixture_worker(project,partial_cfg,3,'resumed')
    if not go2_nested_equal(go2_k21_learner_scientific_state(full),go2_k21_learner_scientific_state(resumed)):raise AssertionError('Fresh-kernel exact learner recovery differs')
    stop_cfg={**copy.deepcopy(config),'run_id':'k21-learner-fixture-'+token+'-earlystop','optimizer_steps':10,'minimum_improvement':10.,'patience_evaluations':2}
    stopped=go2_k21_learner_fixture_worker(project,stop_cfg,10,'earlystop')
    if stopped['global_step']!=4 or stopped['metrics']['nonimproving_evaluations']!=2 or stopped['stop_reason']!='DEVELOPMENT_EARLY_STOP':raise AssertionError('Development patience did not stop at the registered boundary')
    saved,_=go2_load_checkpoint(project/'checkpoints'/stop_cfg['run_id'],which='best')
    if saved['global_step']!=0:raise AssertionError('Step-zero best was discarded despite no development improvement')
    again=go2_k21_learner_fixture_worker(project,stop_cfg,10,'repeat_stopped')
    if not go2_nested_equal(go2_k21_learner_scientific_state(stopped),go2_k21_learner_scientific_state(again)):raise AssertionError('Run All reset early-stop patience or performed extra updates')
    directory=project/'checkpoints'/partial_cfg['run_id'];manifest,states=go2_manifest_states(directory)
    if len(states)<2 or manifest['best'] not in states:raise AssertionError('Latest-two/best retention failed')
    restored,metadata=go2_k21_load_player(project,partial_cfg['run_id'],which='latest',device='cpu')
    if go2_tensor_digest(restored.state_dict())!=go2_tensor_digest(resumed['model']):raise AssertionError('Frozen player loader changed the model')
    wrong=copy.deepcopy(resumed['identity']);wrong['buffer_rows_sha256']='0'*64
    try:go2_load_checkpoint(directory,wrong)
    except RuntimeError:pass
    else:raise AssertionError('Changed immutable buffer was accepted')
    with go2_checkpoint_lock(directory):
        try:
            with go2_checkpoint_lock(directory):pass
        except RuntimeError:pass
        else:raise AssertionError('Second checkpoint writer was accepted')
    report={'schema':'ahr-k21-learner-validation-v1','all_passed':True,'updated_at_utc':go2_now(),'sources':go2_k21_sources(project),
        'device':device,'status':'PASS','production_training_started':False,'exact_scientific_state':True,'fresh_kernel_per_invocation':True,
        'continuous_run_id':config['run_id'],'resumed_run_id':partial_cfg['run_id'],'earlystop_run_id':stop_cfg['run_id'],
        'checks':['teacher-only Worker and auxiliary targets preserved','optional outcome term isolated','D4 policy/legal alignment',
            'exact model/optimizer/scheduler/scaler/RNG/sampler/metrics resume','step-zero best retained','early-stop patience survives resume',
            'latest two and best retained','immutable-buffer mismatch refused','single writer enforced','frozen player weights verified'],
        'completed_optimizer_steps':6,'earlystop_optimizer_steps':4,'retained_checkpoints':len(states)}
    go2_atomic_json(project/'reports/ahr_v2/k21_learner_validation.json',report);return report


def go2_k21_learner_validate(project=None,device='cpu'):
    return go2_k21_validate_learner(project,device=device)


## Verified promotion and a resumable human game

Definition cells are imported only through the `ahr-export` tag. Production work starts at the explicit Run All launcher.

In [ ]:
GO2_K21_PROMOTION_POLICY={'min_kata_games':100,'min_kata_wins':55,'min_gnu_games':20,
    'min_gnu_win_rate':.70,'max_gnu_drop':.10,'max_unknown':0,'ahr_soft_move_seconds':2.0,'min_opening_plies':8}

def _go2_k21_snapshot_model(project,path,sha,model_type,device='cpu'):
    project=go2_project(project);path=Path(path)
    if not path.is_file() or go2_file_sha(path)!=sha:raise RuntimeError('Notebook21 promoted snapshot checksum mismatch')
    state=torch.load(path,map_location='cpu',weights_only=False)
    if not isinstance(state,dict) or GO2_REQUIRED_STATE-set(state) or state['schema']!=GO2_SCHEMA:raise RuntimeError('Promoted snapshot is not a complete saved checkpoint')
    if state['identity'].get('training_schema')!=GO2_K21_LEARNER_SCHEMA:
        return _go2_k20_snapshot_model(project,path,sha,model_type,device=device)
    if model_type!='Go2Network' or state['identity'].get('model_type')!='Go2Network':raise RuntimeError('Notebook21 ordinary network model type differs')
    if state['identity'].get('sources')!=go2_k21_sources(project):raise RuntimeError('Promoted notebook21 source identity changed')
    if state['identity'].get('config')!=state['config']:raise RuntimeError('Promoted notebook21 configuration differs from checkpoint identity')
    with torch.random.fork_rng(devices=[]):model=Go2Network(Go2NetConfig(**state['identity']['network']))
    model.load_state_dict(state['model'],strict=True);model.to(device).eval();model.requires_grad_(False)
    return model,state

def _go2_k21_read_promotion_pointer(project,path=None):
    project=go2_project(project);path=Path(path or project/'reports/ahr_v2/k21_promoted_player.json')
    if not path.exists():return None
    try:value=json.loads(path.read_text())
    except (ValueError,OSError) as exc:raise RuntimeError('Notebook21 promotion pointer is unreadable; no silent fallback') from exc
    signed=dict(value);checksum=signed.pop('pointer_sha256',None)
    if value.get('schema')!='ahr_k21_promoted_player_v1' or checksum!=go2_digest(signed):raise RuntimeError('Notebook21 promotion pointer checksum/schema mismatch; no silent fallback')
    if value.get('sources')!=go2_k21_sources(project):raise RuntimeError('Notebook21 promoted player source identity differs')
    snapshot=(project/value['snapshot_file']).resolve()
    if not snapshot.is_relative_to((project/'checkpoints/ahr-v2-k21-promoted-players').resolve()):raise RuntimeError('Notebook21 promoted snapshot must stay in its dedicated directory')
    if not snapshot.is_file() or go2_file_sha(snapshot)!=value['snapshot_sha256']:raise RuntimeError('Notebook21 promoted snapshot checksum mismatch')
    if value.get('model_type') not in ('Go2Network','Go2K20Network'):raise RuntimeError('Unsupported promoted model type')
    if value.get('metadata',{}).get('model_identity')!=value.get('model_identity'):raise RuntimeError('Promotion pointer metadata identity differs')
    return value

def go2_k21_promote(project,run_id,model,metadata,evidence,policy=None):
    project=go2_project(project)
    if not run_id or Path(run_id).name!=run_id:raise ValueError('Simple local promotion run ID required')
    policy=copy.deepcopy(GO2_K21_PROMOTION_POLICY)|copy.deepcopy(policy or {})
    if set(policy)!=set(GO2_K21_PROMOTION_POLICY):raise ValueError('Unknown notebook21 promotion policy setting')
    if policy['min_kata_games']<100 or policy['min_kata_wins']<55 or policy['min_gnu_games']<20 or policy['min_gnu_win_rate']<.70 or policy['max_gnu_drop']>.10 or policy['max_unknown']!=0 or policy['ahr_soft_move_seconds']!=2.0 or policy['min_opening_plies']<8:
        raise ValueError('Notebook21 promotion cannot weaken the registered 55/100 KataGo and GNU regression gate')
    if metadata.get('is_smoke') or metadata.get('optimizer_step',0)<=0:raise RuntimeError('Disposable or unchanged step-zero candidates cannot be promoted')
    model_identity=go2_model_identity(model)
    if model_identity!=metadata.get('model_identity'):raise RuntimeError('Candidate weights differ from evaluated metadata')
    result=go2_k20_promotion_decision(project,evidence,model_identity,policy,metadata);result['promoted']=False
    # The registered confirmation uses exactly the declared number of games.
    if result['kata_games']!=policy['min_kata_games']:
        result['decision']='KEEP_CHAMPION';result['reasons'].append('Confirmation game count differs from the fixed registered count')
    report_path=project/'reports/ahr_v2'/run_id/'promotion.json';go2_atomic_json(report_path,result)
    if result['decision']!='PASS':return result
    if not metadata.get('run_id') or Path(metadata['run_id']).name!=metadata['run_id']:raise ValueError('Invalid candidate run path')
    source=(project/'checkpoints'/metadata['run_id']/metadata['checkpoint_file']).resolve()
    if not source.is_relative_to((project/'checkpoints'/metadata['run_id']).resolve()):raise ValueError('Candidate checkpoint escapes its run')
    kind=metadata.get('model_type','Go2Network');verified,state=_go2_k21_snapshot_model(project,source,metadata['checkpoint_sha256'],kind,'cpu')
    if state['config'].get('is_smoke') or state['config'].get('purpose')=='validation_fixture':raise RuntimeError('Synthetic checkpoint cannot be promoted')
    if go2_model_identity(verified)!=model_identity or state['global_step']!=metadata['optimizer_step']:raise RuntimeError('Candidate snapshot differs from the evaluated model or optimizer step')
    directory=project/'checkpoints/ahr-v2-k21-promoted-players'
    with go2_checkpoint_lock(directory):
        existing=_go2_k21_read_promotion_pointer(project)
        if existing is not None:_go2_k21_snapshot_model(project,project/existing['snapshot_file'],existing['snapshot_sha256'],existing['model_type'])
        snapshot=directory/('snapshot-'+model_identity[:24]+'.pt')
        if snapshot.exists() and go2_file_sha(snapshot)!=metadata['checkpoint_sha256']:raise RuntimeError('Notebook21 promoted snapshot identity collision')
        if not snapshot.exists():go2_atomic_bytes(snapshot,source.read_bytes())
        if go2_file_sha(snapshot)!=metadata['checkpoint_sha256']:raise RuntimeError('Promoted checkpoint copy hash verification failed')
        pointer={'schema':'ahr_k21_promoted_player_v1','run_id':run_id,'model_type':kind,'model_identity':model_identity,
            'snapshot_file':str(snapshot.relative_to(project)),'snapshot_sha256':metadata['checkpoint_sha256'],
            'metadata':copy.deepcopy(metadata),'evidence':result,'sources':go2_k21_sources(project),'promoted_at_utc':go2_now()}
        pointer['pointer_sha256']=go2_digest(pointer);go2_atomic_json(project/'reports/ahr_v2/k21_promoted_player.json',pointer)
    result=result|{'promoted':True,'pointer':'reports/ahr_v2/k21_promoted_player.json'};go2_atomic_json(report_path,result)
    return result

def go2_k21_load_verified_player(project,device='cpu',allow_smoke=False):
    project=go2_project(project);pointer=_go2_k21_read_promotion_pointer(project)
    if pointer is None:return go2_k20_load_promoted_player(project,device=device,allow_smoke=allow_smoke)
    model,state=_go2_k21_snapshot_model(project,project/pointer['snapshot_file'],pointer['snapshot_sha256'],pointer['model_type'],device)
    metadata=copy.deepcopy(pointer['metadata'])
    if (metadata.get('is_smoke') or state['config'].get('is_smoke') or state['config'].get('purpose')=='validation_fixture') and not allow_smoke:raise RuntimeError('Disposable checkpoint cannot be a human opponent')
    if go2_model_identity(model)!=pointer['model_identity']:raise RuntimeError('Promoted notebook21 model identity differs')
    metadata.update(label='Promoted AHR player — '+metadata['run_id'],promotion_status='PASS',model_type=pointer['model_type'],
        best_selection_basis=f"Fresh KataGo1 confirmation {pointer['evidence']['kata_wins']}/{pointer['evidence']['kata_games']}; GNU regression passed; development evidence")
    return model,metadata


In [ ]:
def go2_k21_play_namespace(project):
    project=go2_project(project);namespace=go2_k18_namespace(project)
    namespace['go2_import_exports'](project/'experiments/ahr_v2/19_stronger_scout_worker_training.ipynb',namespace)
    namespace['go2_import_exports'](project/'experiments/ahr_v2/20_economic_rewards_and_teacher_guided_improvement.ipynb',namespace)
    base=namespace['Go2TeacherPlaySession'];base_ui=namespace['Go2TeacherPlayUI']
    source_fn=lambda p:go2_k21_sources(p)
    namespace['go2_teacher_play_sources']=source_fn
    namespace['go2_teacher_play_load_model']=lambda p,which='best',device='cpu',allow_smoke=False:go2_k21_load_verified_player(p,device,allow_smoke)
    def factory(config,kind):
        if kind not in ('Go2Network','Go2K20Network'):raise RuntimeError('Unsupported frozen human opponent model type')
        cls=namespace['Go2K20Network'] if kind=='Go2K20Network' else namespace['Go2Network']
        with torch.random.fork_rng(devices=[]):return cls(namespace['Go2NetConfig'](**config))
    class Session(base):
        def __init__(self,project,model,metadata,human_color=1,search_config=None,game_id=None,persist=True):
            config=search_config or namespace['Go2SearchConfig'](simulations=512,batch_size=8,wall_time_seconds=2.0)
            super().__init__(project,model,metadata,human_color,config,game_id,persist)
            self.directory=self.project/'checkpoints/ahr-v2-k21-human-games'/self.game_id
        def save(self):
            if not self._persist:return
            self.directory=self.project/'checkpoints/ahr-v2-k21-human-games'/self.game_id
            namespace['go2_teacher_play_verify_replay'](self.state)
            if source_fn(self.project)!=self.sources:raise RuntimeError('Human player source definitions changed during this game.')
            path=self.directory/'game.json'
            with namespace['go2_checkpoint_lock'](self.directory):
                previous=json.loads(path.read_text()) if path.exists() else None
                if previous is not None:
                    signed=dict(previous);checksum=signed.pop('payload_sha256',None)
                    if checksum!=go2_digest(signed):raise RuntimeError('Existing human game checksum changed')
                    if previous.get('revision')!=self.revision:raise RuntimeError('Another session changed this game.')
                if not path.exists() and self.revision:raise RuntimeError('Saved game disappeared; no silent restart.')
                snapshot=self.directory/'opponent.pt'
                if previous is not None and (not snapshot.is_file() or go2_file_sha(snapshot)!=previous['snapshot_sha256']):raise RuntimeError('Existing frozen opponent snapshot changed')
                if snapshot.exists() and previous is None:
                    orphan=torch.load(snapshot,map_location='cpu',weights_only=True)
                    if orphan.get('schema')!='ahr_k21_human_snapshot_v1':raise RuntimeError('Orphan human snapshot schema differs')
                    frozen=factory(orphan['config'],orphan['model_type']);frozen.load_state_dict(orphan['model'],strict=True)
                    if go2_model_identity(frozen)!=self.metadata['model_identity']:raise RuntimeError('Orphan human snapshot differs from this opponent')
                if not snapshot.exists():
                    payload={'schema':'ahr_k21_human_snapshot_v1','model_type':self.metadata.get('model_type','Go2Network'),
                        'config':dataclasses.asdict(self.model.config),'model':{k:v.detach().cpu().clone() for k,v in self.model.state_dict().items()}}
                    stream=io.BytesIO();torch.save(payload,stream);go2_atomic_bytes(snapshot,stream.getvalue())
                if go2_model_identity(self.model)!=self.metadata['model_identity']:raise RuntimeError('Frozen opponent weights changed during the game')
                record={'schema':'ahr_k21_human_game_v1','game_id':self.game_id,'revision':self.revision+1,'sources':self.sources,
                    'model_identity':self.metadata['model_identity'],'snapshot_file':'opponent.pt','snapshot_sha256':go2_file_sha(snapshot),
                    'metadata':self.metadata,'human_color':self.human_color,'search_config':dataclasses.asdict(self.search_config),
                    'state':namespace['go2_state_to_dict'](self.state),'state_sha256':namespace['go2_state_digest'](self.state),
                    'events':self.events,'last_search':self.last_search,'updated_at_utc':go2_now()}
                record=json.loads(json.dumps(record,default=go2_json_default,allow_nan=False));record['payload_sha256']=go2_digest(record)
                go2_atomic_json(path,record);self.revision=record['revision']
                if not self.metadata.get('is_smoke'):go2_atomic_json(self.project/'reports/ahr_v2/k21_latest_human_game.json',{'game_path':str(path.relative_to(self.project)),'updated_at_utc':record['updated_at_utc']})
        @classmethod
        def restore(cls,project,game_path,device='cpu'):
            project=Path(project);path=Path(game_path);path=path if path.is_absolute() else project/path;path=path.resolve()
            if not path.is_relative_to((project/'checkpoints/ahr-v2-k21-human-games').resolve()) or path.name!='game.json':raise ValueError('Resume a saved notebook21 AHR game.')
            record=json.loads(path.read_text());checksum=record.pop('payload_sha256',None)
            if checksum!=go2_digest(record) or record.get('schema')!='ahr_k21_human_game_v1':raise RuntimeError('Human game record checksum/schema mismatch.')
            if record['game_id']!=path.parent.name:raise RuntimeError('Saved human game ID/path differs')
            if record['sources']!=source_fn(project):raise RuntimeError('Human-game sources changed; resume refused.')
            snapshot=path.parent/'opponent.pt'
            if record['snapshot_file']!='opponent.pt' or go2_file_sha(snapshot)!=record['snapshot_sha256']:raise RuntimeError('Frozen human opponent snapshot corrupt.')
            payload=torch.load(snapshot,map_location='cpu',weights_only=True)
            if payload['schema']!='ahr_k21_human_snapshot_v1':raise RuntimeError('Frozen human opponent schema mismatch.')
            model=factory(payload['config'],payload['model_type']);model.load_state_dict(payload['model'],strict=True);model.to(device).eval();model.requires_grad_(False)
            if go2_model_identity(model)!=record['model_identity']:raise RuntimeError('Frozen human opponent model identity differs.')
            state=namespace['go2_state_from_dict'](record['state']);namespace['go2_teacher_play_verify_replay'](state)
            if namespace['go2_state_digest'](state)!=record['state_sha256']:raise RuntimeError('Saved human Go history differs.')
            if len(record['events'])!=len(state.actions) or any(int(event['action'])!=int(action) for event,action in zip(record['events'],state.actions)):raise RuntimeError('Saved event list differs from the legal replay')
            if record['metadata'].get('model_identity')!=record['model_identity']:raise RuntimeError('Saved game metadata identity differs')
            session=cls(project,model,record['metadata'],human_color=record['human_color'],search_config=namespace['Go2SearchConfig'](**record['search_config']),game_id=record['game_id'],persist=False)
            session.directory=path.parent;session.state=state;session.events=record['events'];session.last_search=record['last_search'];session.revision=record['revision'];session._persist=True
            return session
    namespace['Go2TeacherPlaySession']=Session
    class UI(base_ui):
        def __init__(self,*args,**kwargs):
            project=Path(args[0] if args else kwargs['project'])
            path=project/'reports/ahr_v2/k21_latest_human_game.json'
            previous_game=json.loads(path.read_text())['game_path'] if path.exists() else None
            super().__init__(*args,**kwargs)
            self.resume_path=previous_game
            self.which.layout.display='none';self.which.description='Verified player';self.visits.value=512
            self.buttons['Reload checkpoint'].description='Reload verified player';self.buttons['Reload checkpoint'].layout.width='190px'
            self.widget.children[0].value='<h3 style="margin:0">Play verified AHR · 9×9 · notebook 21</h3>'
    namespace['Go2TeacherPlayUI']=UI
    return namespace

def go2_k21_play(project,device='cpu',persist=True,display_widget=True):
    project=go2_project(project);model,metadata=go2_k21_load_verified_player(project,device)
    namespace=go2_k21_play_namespace(project);ui=namespace['Go2TeacherPlayUI'](project,model,metadata,device=device,persist=persist)
    if display_widget:
        from IPython.display import display
        display(ui.widget)
    return ui



In [ ]:
def go2_k21_player_validate(project=None):
    project=go2_project(project);token=uuid.uuid4().hex[:10]
    production_paths=[project/'reports/ahr_v2/k20_promoted_player.json',project/'reports/ahr_v2/k21_promoted_player.json',project/'reports/ahr_v2/k20_latest_human_game.json',project/'reports/ahr_v2/k21_latest_human_game.json']
    previous={str(path):go2_file_sha(path) if path.is_file() else None for path in production_paths}
    namespace=go2_k21_play_namespace(project)
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(212121);model=namespace['Go2Network'](namespace['Go2NetConfig'](board_size=9,width=8,blocks=1))
    model.eval();model.requires_grad_(False);model_identity=go2_model_identity(model)
    metadata={'model_identity':model_identity,'model_type':'Go2Network','is_smoke':True,'optimizer_step':0,'label':'Disposable notebook21 restoration fixture','run_id':'k21-player-fixture-'+token}
    session=namespace['Go2TeacherPlaySession'](project,model,metadata,game_id='k21-play-fixture-'+token,persist=True)
    session.move(39);session.human_color=0;session.move(41,actor='ahr')
    restored=namespace['Go2TeacherPlaySession'].restore(project,session.directory/'game.json')
    if go2_tensor_digest(restored.model.state_dict())!=go2_tensor_digest(model.state_dict()):raise AssertionError('Notebook21 frozen model restoration differs')
    if go2_state_digest(restored.state)!=go2_state_digest(session.state) or restored.events!=session.events:raise AssertionError('Notebook21 human history restoration differs')
    fixture_config=go2_k21_train_config(metadata['run_id'],'synthetic',device='cpu');fixture_config.update(is_smoke=True,fixture_data=True)
    fixture_state={'schema':GO2_SCHEMA,'identity':{'schema':GO2_SCHEMA,'training_schema':GO2_K21_LEARNER_SCHEMA,'sources':go2_k21_sources(project),
        'config':fixture_config,'network':dataclasses.asdict(model.config),'model_type':'Go2Network'},
        'config':fixture_config,'model':copy.deepcopy(model.state_dict()),'optimizer':{},'scheduler':{},'scaler':{},
        'rng':go2_capture_rng(),'sampler':{},'global_step':0,'metrics':{}}
    snapshot=project/'checkpoints/ahr-v2-k21-promoted-players'/('fixture-'+token+'.pt');payload=io.BytesIO();torch.save(fixture_state,payload)
    with go2_checkpoint_lock(snapshot.parent):go2_atomic_bytes(snapshot,payload.getvalue())
    loaded,_=_go2_k21_snapshot_model(project,snapshot,go2_file_sha(snapshot),'Go2Network')
    if go2_model_identity(loaded)!=model_identity:raise AssertionError('Notebook21 checkpoint schema cannot be loaded exactly')
    pointer={'schema':'ahr_k21_promoted_player_v1','sources':go2_k21_sources(project),'model_identity':model_identity,'model_type':'Go2Network',
        'snapshot_file':str(snapshot.relative_to(project)),'snapshot_sha256':go2_file_sha(snapshot),'metadata':metadata,'purpose':'validation_fixture'}
    pointer['pointer_sha256']=go2_digest(pointer);fixture=project/'reports/ahr_v2/fixtures'/('k21-promotion-pointer-'+token+'.json');go2_atomic_json(fixture,pointer)
    if _go2_k21_read_promotion_pointer(project,fixture) is None:raise AssertionError('Valid notebook21 fixture pointer was refused')
    pointer['snapshot_sha256']='0'*64;pointer['pointer_sha256']=go2_digest({k:v for k,v in pointer.items() if k!='pointer_sha256'});go2_atomic_json(fixture,pointer)
    try:_go2_k21_read_promotion_pointer(project,fixture)
    except RuntimeError:pass
    else:raise AssertionError('Corrupt promoted snapshot was accepted')
    pointer['snapshot_sha256']=go2_file_sha(snapshot);pointer['pointer_sha256']='0'*64;go2_atomic_json(fixture,pointer)
    try:_go2_k21_read_promotion_pointer(project,fixture)
    except RuntimeError:pass
    else:raise AssertionError('Corrupt promotion pointer was accepted')
    ui=go2_k21_play(project,device='cpu',persist=False,display_widget=False)
    if len(ui.points)!=81 or ui.visits.value!=512 or ui.seconds.value!=2.0 or ui.which.layout.display!='none':raise AssertionError('Notebook21 human UI is not ready')
    if 'ahr-v2-k21-human-games' not in str(ui.session.directory):raise AssertionError('Notebook21 game directory isolation failed')
    after={str(path):go2_file_sha(path) if path.is_file() else None for path in production_paths}
    if previous!=after:raise AssertionError('Validation changed a production promotion or latest-game pointer')
    kata={'games':100,'wins':55,'unknown':0};gnu={'games':20,'wins':16,'unknown':0}
    if _go2_k20_gate_counts(kata,gnu,gnu,GO2_K21_PROMOTION_POLICY)['decision']!='PASS':raise AssertionError('Qualifying gate fixture failed')
    for failed in (kata|{'wins':54},kata|{'unknown':1}):
        if _go2_k20_gate_counts(failed,gnu,gnu,GO2_K21_PROMOTION_POLICY)['decision']!='KEEP_CHAMPION':raise AssertionError('Invalid gate fixture passed')
    receipt={'schema':'ahr_k21_player_validation_v1','status':'PASS','all_passed':True,'sources':go2_k21_sources(project),'validated_at_utc':go2_now(),
        'notebook21_snapshot_exact_load':True,'human_snapshot_exact_restore':True,'human_history_exact_restore':True,
        'pointer_and_snapshot_corruption_rejected':True,'human_ui_ready':True,'game_directory_isolated':True,'production_pointers_unchanged':True,
        'production_training_started':False,'production_promotion_performed':False,'saved_fixture_game':str((session.directory/'game.json').relative_to(project))}
    go2_atomic_json(project/'reports/ahr_v2/k21_player_validation.json',receipt);return receipt


## Fresh-kernel validation and launch checks

Definition cells are imported only through the `ahr-export` tag. Production work starts at the explicit Run All launcher.

In [ ]:
def go2_k21_state_boundary_validate(project,checkpoint_directory=None):
    """Assert exact chronological conversion across independently imported state classes."""
    project=Path(project)
    namespace=go2_k21_data_namespace(project)
    foreign=namespace['go2_initial_state'](9,7.5)
    for action in (40,10,30,50):foreign=namespace['go2_step'](foreign,action)
    if type(foreign) is Go2State:raise AssertionError('Regression requires two independently exported dataclass definitions.')
    local=go2_state_from_dict(go2_state_to_dict(foreign))
    if local==foreign:raise AssertionError('Regression did not reproduce Python dataclass class-sensitive equality.')
    if any(getattr(local,field.name)!=getattr(foreign,field.name) for field in dataclasses.fields(local)):
        raise AssertionError('Fixture must differ only in its Python class identity.')
    if not namespace['go2_verify_replay'](foreign) or not go2_verify_replay(local):
        raise AssertionError('Each namespace must independently validate its own exact state.')
    teacher=object.__new__(Go2K21Teacher)
    teacher.visits=128
    teacher.identity={'schema':'k21-state-boundary-validation-only','no_native_process':True}
    query,_=teacher._request(foreign,128,[20])
    if query['state']!=go2_state_to_dict(local):raise AssertionError('State boundary changed complete chronology.')
    malformed=dataclasses.replace(foreign,recent_boards=(foreign.board,))
    try:teacher._request(malformed,128,None)
    except ValueError:pass
    else:raise AssertionError('Canonicalization must reject altered chronological frames.')
    checked=0
    if checkpoint_directory is not None:
        saved,_=go2_load_checkpoint(Path(checkpoint_directory))
        if saved is None:raise AssertionError('Requested saved replay regression checkpoint is absent.')
        for row in saved['pending_rows']:
            for name in ('state','next_state'):
                position=namespace['go2_state_from_dict'](row[name])
                if position.terminal:continue
                request,_=teacher._request(position,128,None)
                if request['state']!=row[name]:raise AssertionError('Saved-state canonicalization changed chronology.')
                checked+=1
    return {'status':'PASS','different_export_namespace_classes':True,
        'equal_fields_but_dataclass_equality_false':True,'foreign_state_request_accepted':True,
        'full_schema_preserved':True,'tampered_chronology_rejected':True,
        'saved_live_states_checked':checked,'no_native_process':True}


def go2_k21_fresh_controller_fixture(project, config, max_stages, label):
    """Execute only disposable CPU controller stages in a new project kernel."""
    from nbclient import NotebookClient
    project=go2_project(project)
    if config.get('purpose')!='validation_fixture':
        raise ValueError('A disposable validation_fixture configuration is required.')
    folder=project/'reports/ahr_v2/fixtures/k21-controller'/config['run_id']
    folder.mkdir(parents=True,exist_ok=True)
    path=folder/(label+'.ipynb')
    body='CONFIG=json.loads('+repr(json.dumps(config))+')\n'
    body+='torch.set_num_threads(1)\n'
    body+='STATE=go2_k21_run(PROJECT,CONFIG,max_stages='+repr(max_stages)+',_fixture=True)\n'
    body+="print({'step':STATE['global_step'],'cursor':STATE['sampler']['stage_index']})"
    book=nbformat.v4.new_notebook(cells=[
        nbformat.v4.new_markdown_cell('Disposable CPU controller recovery fixture. No teacher, Go games, production dataset or GPU work.'),
        nbformat.v4.new_code_cell(go2_k21_bootstrap(project)),
        nbformat.v4.new_code_cell(body)],
        metadata={'kernelspec':{'display_name':'AHR Go','language':'python','name':'ahr-go'}})
    nbformat.write(book,path)
    try:
        NotebookClient(book,timeout=180,kernel_name='ahr-go',
            resources={'metadata':{'path':str(project)}}).execute()
    finally:
        nbformat.write(book,path)
    state,entry=go2_load_checkpoint(project/'checkpoints'/config['run_id'])
    if state is None:raise AssertionError('Controller fixture failed to create its checkpoint.')
    return state,entry


def go2_k21_controller_validate(project):
    """Check actual fresh-kernel stage/RNG resume, corrupt state and deliberate stop."""
    project=go2_project(project);token=uuid.uuid4().hex[:10]
    config=go2_k21_config('ahr-v2-k21-controller-fixture-'+token+'-full')
    config['purpose']='validation_fixture'
    count=len(go2_k21_stages(config))
    if count!=4:raise AssertionError('The registered disposable fixture has four stages.')
    full,_=go2_k21_fresh_controller_fixture(project,config,None,'continuous')
    resumed_config=copy.deepcopy(config)
    resumed_config['run_id']='ahr-v2-k21-controller-fixture-'+token+'-resumed'
    prefix,_=go2_k21_fresh_controller_fixture(project,resumed_config,2,'interrupted')
    resumed,entry=go2_k21_fresh_controller_fixture(project,resumed_config,None,'resumed')
    keys=('global_step','model','optimizer','scheduler','scaler','rng','sampler','metrics','results')
    scientific=lambda state:{key:copy.deepcopy(state[key]) for key in keys}
    if not go2_nested_equal(scientific(full),scientific(resumed)):
        raise AssertionError('Notebook 21 fresh-kernel controller/RNG resume changed scientific state.')
    if prefix['sampler']['stage_index']!=2 or resumed['sampler']['stage_index']!=count:
        raise AssertionError('The interrupted or resumed stage cursor is incorrect.')
    directory=project/'checkpoints'/resumed_config['run_id']
    manifest,states=go2_manifest_states(directory)
    if len(states)<2 or manifest['best'] not in states or manifest['latest'] not in states:
        raise AssertionError('Latest-two/best retention is incomplete.')
    wrong=copy.deepcopy(resumed['identity']);wrong['config']['seed']+=1
    try:go2_load_checkpoint(directory,identity=wrong)
    except RuntimeError:pass
    else:raise AssertionError('A changed controller configuration was silently accepted.')
    wrong=copy.deepcopy(resumed['identity'])
    source_name=next(iter(wrong['sources']));wrong['sources'][source_name]='0'*64
    try:go2_load_checkpoint(directory,identity=wrong)
    except RuntimeError:pass
    else:raise AssertionError('Changed notebook definitions were silently accepted.')
    with tempfile.TemporaryDirectory(prefix='k21-controller-corrupt-',dir=project/'checkpoints') as temp:
        corrupt=Path(temp);shutil.copytree(directory,corrupt,dirs_exist_ok=True)
        payload=corrupt/manifest['latest'];raw=bytearray(payload.read_bytes())
        raw[len(raw)//2]^=1;payload.write_bytes(raw)
        try:go2_load_checkpoint(corrupt)
        except RuntimeError:pass
        else:raise AssertionError('Corrupt controller checkpoint was silently restarted.')
    stopped=copy.deepcopy(config);stopped['run_id']='ahr-v2-k21-controller-fixture-'+token+'-stop'
    marker=project/'checkpoints'/stopped['run_id']/'STOP';marker.parent.mkdir(parents=True,exist_ok=True)
    marker.write_text('Disposable checkpoint-boundary stop validation.\n')
    paused,_=go2_k21_fresh_controller_fixture(project,stopped,None,'stopped')
    if paused['sampler']['stage_index']!=0:
        raise AssertionError('Stopped controller advanced its stage cursor.')
    marker.unlink()
    after_stop,_=go2_k21_fresh_controller_fixture(project,stopped,None,'resume-after-stop')
    if not go2_nested_equal(scientific(full),scientific(after_stop)):
        raise AssertionError('An explicit stop/resume changed scientific state.')
    return {'status':'PASS','fresh_kernel_exact_stage_rng_resume':True,
        'configuration_mismatch_refused':True,'source_mismatch_refused':True,
        'corruption_refused':True,'stop_preserves_stage_cursor':True,
        'latest_two_and_best_valid':True,'completed_fixture_stages':count,
        'resumed_checkpoint':entry,'production_started':False}


def go2_k21_detached_fixture_validate(project):
    """Run four disposable CPU stages in a new tmux session, with no production launcher."""
    project=go2_project(project)
    if shutil.which('tmux') is None:raise RuntimeError('Detached fixture requires tmux.')
    token=uuid.uuid4().hex[:10];session='ahr-k21-fixture-'+token
    config=go2_k21_config('ahr-v2-k21-detached-fixture-'+token)
    config['purpose']='validation_fixture'
    folder=project/'reports/ahr_v2/fixtures/k21-controller'/config['run_id']
    folder.mkdir(parents=True,exist_ok=True)
    worker_path=folder/'detached.ipynb';result_path=folder/'returned.json';log_path=folder/'tmux.log'
    body='CONFIG=json.loads('+repr(json.dumps(config))+')\n'
    body+='STATE=go2_k21_run(PROJECT,CONFIG,_fixture=True)\n'
    body+='go2_atomic_json(Path('+repr(str(result_path))+'),{"status":"RETURNED","purpose":"validation_fixture","stages":STATE["sampler"]["stage_index"]})'
    book=nbformat.v4.new_notebook(cells=[
        nbformat.v4.new_markdown_cell('Detached CPU checkpoint fixture. No Go games, production data, GPU work or production launcher.'),
        nbformat.v4.new_code_cell(go2_k21_bootstrap(project)),nbformat.v4.new_code_cell(body)],
        metadata={'kernelspec':{'display_name':'AHR Go','language':'python','name':'ahr-go'}})
    nbformat.write(book,worker_path)
    args=[str(project/'.venv/bin/python'),'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace',
        '--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=120',str(worker_path)]
    command=shlex.join(args)+' > '+shlex.quote(str(log_path))+' 2>&1'
    registration={'session':session,'purpose':'validation_fixture','run_id':config['run_id'],
        'sources':go2_k21_sources(project),'production_started':False}
    go2_atomic_json(folder/'launch.json',registration)
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(project),command],check=True)
    deadline=time.monotonic()+45
    while not result_path.exists() and time.monotonic()<deadline:time.sleep(.25)
    if not result_path.exists():raise AssertionError('Detached CPU fixture did not finish; inspect '+str(log_path))
    result=json.loads(result_path.read_text())
    if result!={'status':'RETURNED','purpose':'validation_fixture','stages':4}:
        raise AssertionError('Detached fixture returned unexpected stage state.')
    state,entry=go2_load_checkpoint(project/'checkpoints'/config['run_id'])
    if state is None or state['sampler']['stage_index']!=4:
        raise AssertionError('Detached fixture did not commit all four stages.')
    return {'status':'PASS','session':session,'purpose':'validation_fixture','completed_stages':4,
        'checkpoint':entry,'production_started':False,'registered_worker':str(worker_path.relative_to(project))}


def go2_k21_gate_validate():
    base={'attempts':40,'WIN':3,'UNKNOWN':0}
    if go2_k21_select_screen(base,{'attempts':40,'WIN':6,'UNKNOWN':0})['decision']!='KEEP':
        raise AssertionError('A below-threshold development gain advanced.')
    if go2_k21_select_screen(base,{'attempts':40,'WIN':7,'UNKNOWN':0})['decision']!='ADVANCE':
        raise AssertionError('The registered development gain threshold was not applied.')
    if go2_k21_select_screen(base,{'attempts':40,'WIN':8,'UNKNOWN':1})['decision']!='KEEP':
        raise AssertionError('An incomplete screen with UNKNOWN outcomes advanced.')
    try:go2_k21_select_screen(base,{'attempts':38,'WIN':8,'UNKNOWN':0})
    except ValueError:pass
    else:raise AssertionError('Unmatched screening denominators were accepted.')
    config=go2_k21_config()
    if config['promotion_policy']['min_kata_games']!=100 or config['promotion_policy']['min_kata_wins']!=55:
        raise AssertionError('The registered 55/100 fresh-confirmation gate changed.')
    return {'status':'PASS','unknown_screen_rejected':True,'unequal_denominators_rejected':True,
        'registered_gain_threshold_applied':True,'fresh_confirmation_minimum':'55/100'}


def _go2_k21_assert_validation_passed(result, label):
    if not isinstance(result,dict):raise AssertionError(label+' did not return a validation record.')
    if result.get('status')!='PASS' and result.get('all_passed') is not True:
        raise AssertionError(label+' validation failed: '+str(result))
    return result


def go2_k21_validate(project, require_native=False, config=None):
    """Source-pinned disposable smoke checks; native GPU tests run only on explicit request."""
    project=go2_project(project);sources=go2_k21_sources(project)
    config=copy.deepcopy(config or go2_k21_config())
    root=project/'reports/ahr_v2';path=root/'k21_validation_cpu.json'
    receipt=json.loads(path.read_text()) if path.is_file() else None
    if not receipt or receipt.get('status')!='PASS' or receipt.get('sources')!=sources:
        boundary=go2_k21_state_boundary_validate(project,
            project/'checkpoints/ahr-v2-k21-20261014-full-v2-cycle-01-buffer-shard-0000-collection')
        gates=go2_k21_gate_validate()
        controller=_go2_k21_assert_validation_passed(go2_k21_controller_validate(project),'Controller')
        detached=_go2_k21_assert_validation_passed(go2_k21_detached_fixture_validate(project),'Detached CPU worker')
        data=_go2_k21_assert_validation_passed(go2_k21_data_validate(project,require_native=False),'Data')
        learner=_go2_k21_assert_validation_passed(go2_k21_learner_validate(project,device='cpu'),'Learner')
        player=_go2_k21_assert_validation_passed(go2_k21_player_validate(project),'Player') if 'go2_k21_player_validate' in globals() else {'status':'NOT_AVAILABLE'}
        if go2_k21_sources(project)!=sources:
            raise RuntimeError('Notebook sources changed during validation; no valid receipt can be saved.')
        receipt={'schema':'ahr-v2-k21-validation-v1','status':'PASS','sources':sources,
            'state_boundary':boundary,'controller':controller,'data':data,'learner':learner,'gates':gates,'detached':detached,'player':player,'production_started':False,
            'native_gpu_validated':False,'validated_at_utc':go2_now()}
        go2_atomic_json(path,receipt)
    if not require_native:return copy.deepcopy(receipt)
    native_path=root/'k21_validation_native.json'
    native=json.loads(native_path.read_text()) if native_path.is_file() else None
    if native and native.get('status')=='PASS' and native.get('sources')==sources and native.get('validation_config_sha256')==go2_digest(config):
        return copy.deepcopy(native)
    data=_go2_k21_assert_validation_passed(go2_k21_data_validate(project,require_native=True,config=config),'Native teacher/data')
    device=config['device']
    learner=_go2_k21_assert_validation_passed(go2_k21_learner_validate(project,device=device),'Native learner')
    if go2_k21_sources(project)!=sources:
        raise RuntimeError('Notebook sources changed during native validation.')
    native={'schema':'ahr-v2-k21-validation-v1','status':'PASS','sources':sources,
        'cpu_receipt_sha256':go2_file_sha(path),'data':data,'learner':learner,
        'validation_config_sha256':go2_digest(config),'validated_device':device,
        'native_gpu_validated':True,'production_started':False,'validated_at_utc':go2_now()}
    go2_atomic_json(native_path,native)
    return native


## Run settings

The default is a bounded two-cycle experiment. Edit settings **before the first launch**. A launched run pins its full configuration; changing it requires a new run ID. `RESUME_STOPPED_RUN` is only for resuming an intentional checkpointed stop.

The CPU smoke tests use tiny disposable models. Native GPU teacher and learner recovery tests run inside the worker, after GPUs are free, before any long stage. If a native exact-resume test fails, the worker stops and records the error.


In [ ]:
RUN_CONFIG=go2_k21_config()
TMUX_SESSION='ahr-v2-k21-gpu0'
RESUME_STOPPED_RUN=False
# Optional edits belong here, before this run has ever launched.
display(pd.DataFrame([{'Run ID':RUN_CONFIG['run_id'],'Cycles':RUN_CONFIG['cycles'],
    'Games per cycle':RUN_CONFIG['collection_games'],'Target labels per cycle':RUN_CONFIG['label_rows_target'],
    'Max updates per cycle':RUN_CONFIG['optimizer_steps'],'Economics':'off','Worker outcome loss':'off'}]))


## Evidence motivating this change

The previous candidate won 3/100 against KataGo at one visit. On the new paired GNU Go level-10 check it won 16/20, versus the incumbent’s 17/20. The small GNU difference is descriptive, not evidence of a reliable strength difference. Both guided variants had their best held-out loss at step zero; the figure uses their shared validation population and objective.


In [ ]:
get_ipython().run_line_magic('matplotlib','inline')
import matplotlib.pyplot as plt
HISTORICAL=json.loads((PROJECT/'reports/ahr_v2/ahr-v2-katago-econ-20261011-v1/results.json').read_text())['results']
RECOVERY=json.loads((PROJECT/'reports/ahr_v2/ahr-v2-katago-econ-20261011-v1-eval-recovery-v1/results.json').read_text())['results']
plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False,'axes.titleweight':'bold'})
fig,axes=plt.subplots(1,2,figsize=(12,4.2),layout='constrained')
for key,label,color,marker in [('guided_control','Teacher control','#2563a6','o'),('guided_economics','Economics variant','#b66b12','s')]:
    points=HISTORICAL[key]['metrics']['development_history']
    axes[0].plot([p['optimizer_step'] for p in points],[p['joint_loss'] for p in points],label=label,color=color,marker=marker)
axes[0].set(title='Previous guided training: held-out loss',xlabel='Optimizer updates',ylabel='Joint validation loss (lower is better)')
axes[0].legend(frameon=False);axes[0].grid(axis='y',alpha=.18)
labels=['Existing player','New candidate'];summaries=[RECOVERY['incumbent_gnu']['summary'],RECOVERY['candidate_gnu']['summary']]
rates=[100*x['WIN']/x['attempts'] for x in summaries]
axes[1].barh(labels,rates,color=['#707780','#2563a6'],height=.55)
for i,(rate,item) in enumerate(zip(rates,summaries)):
    axes[1].text(rate+1,i,str(item['WIN'])+'/'+str(item['attempts'])+' ('+str(round(rate))+'%)',va='center')
axes[1].set(xlim=(0,108),xlabel='Wins (%)',title='GNU Go level 10 · paired development games')
axes[1].set_xticks([0,25,50,75,100]);axes[1].grid(axis='x',alpha=.18);axes[1].set_axisbelow(True)
plt.show()
display(Markdown('**Historical evidence only.** No notebook-21 strength result is available until the new run finishes.'))


## Validate, then launch

Run All validates the CPU recovery paths, then launches the detached worker. Repeated Run All detects a live or completed run. `validate` and `status` modes are used for authoring checks and never launch production.

In [ ]:
if MODE!='status':
    import contextlib
    _validation_log=PROJECT/'reports/ahr_v2/k21_validation/notebook-validation.log'
    _validation_log.parent.mkdir(parents=True,exist_ok=True)
    with _validation_log.open('a') as _log,contextlib.redirect_stdout(_log),contextlib.redirect_stderr(_log):
        VALIDATION=go2_k21_validate(PROJECT,require_native=False,config=RUN_CONFIG)
    display(Markdown('**CPU validation: '+VALIDATION['status']+'**. Native GPU recovery is checked by the detached worker before long stages.'))
if MODE=='run':
    LAUNCH=go2_k21_launch(PROJECT,RUN_CONFIG,session=TMUX_SESSION,resume_stopped=RESUME_STOPPED_RUN)
    display(Markdown('**'+LAUNCH['status']+'** · `'+TMUX_SESSION+'`'))
    display(Markdown('Attach: `tmux attach -t '+TMUX_SESSION+'` · detach: **Ctrl-b**, then **d**.'))
else:
    display(Markdown('**'+MODE+' mode:** the full notebook-21 run has not been launched by this execution.'))


## Progress

Refresh reads saved reports without running another experiment. Stop requests take effect after the active stage saves a checkpoint. Collection, actual optimizer time, matched screens and confirmation are reported separately.

In [ ]:
import html as _html
PROGRESS=widgets.Output()
REFRESH=widgets.Button(description='Refresh progress',icon='refresh')
STOP=widgets.Button(description='Request checkpointed stop',icon='stop',button_style='warning',layout=widgets.Layout(width='260px'))
def show_k21_progress(_=None):
    with PROGRESS:
        clear_output(wait=True)
        status=go2_k21_status(PROJECT,RUN_CONFIG['run_id'])
        display(HTML('<b>'+_html.escape(status['status'])+'</b> · stage: '+_html.escape(str(status.get('stage') or '—'))))
        if status.get('updated_at_utc'):display(HTML('Last saved update: '+_html.escape(status['updated_at_utc'])))
        if status.get('resources'):
            r=status['resources'];display(Markdown('Waiting for resources. Active GPU processes: '+str(r['foreign_gpu_pids'])+'; reserved sessions: '+', '.join(r['reserved_sessions'])+'.'))
        if status.get('error'):display(Markdown('**Stopped on error:** `'+status['error']+'`'))
        rows=[]
        for path in sorted((PROJECT/'reports/ahr_v2').glob(RUN_CONFIG['run_id']+'*/status.json')):
            child=json.loads(path.read_text())
            if child.get('run_id')==RUN_CONFIG['run_id']:continue
            rows.append({'Stage/run':path.parent.name.replace(RUN_CONFIG['run_id']+'-',''),'Status':child.get('status'),
                'Updates':child.get('optimizer_step'),'Labels':child.get('labelled_rows'),
                'Active optimizer min':round(child['active_optimizer_seconds']/60,1) if 'active_optimizer_seconds' in child else None})
        if rows:display(pd.DataFrame(rows).tail(12))
        games=[]
        for path in sorted((PROJECT/'reports/ahr_v2/benchmarks').glob(RUN_CONFIG['run_id']+'*/report.json')):
            report=json.loads(path.read_text());summary=report.get('summary',{})
            games.append({'Match':path.parent.name.replace(RUN_CONFIG['run_id']+'-',''),'Status':report.get('status'),
                'Wins':summary.get('WIN'),'Games':summary.get('attempts'),'UNKNOWN':summary.get('UNKNOWN')})
        if games:display(pd.DataFrame(games))
        result_path=PROJECT/'reports/ahr_v2'/RUN_CONFIG['run_id']/'results.json'
        if result_path.exists():
            results=json.loads(result_path.read_text())['results']
            for key,result in results.items():
                if key.startswith('screen_'):display(Markdown('**'+key+': '+result['decision']+'** · '+result.get('reason','matched development screen')))
            if 'promotion' in results:display(Markdown('**Final selection:** '+results['promotion']['decision']))
        if not rows and not games:display(HTML('No full-run results yet. Existing validated player remains available below.'))
def stop_k21(_):
    go2_k21_stop(PROJECT,RUN_CONFIG['run_id'])
    with PROGRESS:display(Markdown('**Stop requested.** Resume with `RESUME_STOPPED_RUN=True` and Run All.'))
REFRESH.on_click(show_k21_progress);STOP.on_click(stop_k21)
display(widgets.VBox([widgets.HBox([REFRESH,STOP]),PROGRESS]))
show_k21_progress()


## Play the verified player

This uses the notebook-21 candidate only after it passes the confirmation and regression gates. Until then it uses the existing verified player. KataGo does not choose AHR moves during human play. Human games have their own saved board history and opponent snapshot; use Resume to continue the same game.

In [ ]:
display(HTML("""<style>.ahr-go-point.widget-button {margin:0!important;border:0!important;border-radius:0!important;background-image:linear-gradient(#76522d,#76522d),linear-gradient(#76522d,#76522d)!important;background-size:100% 1px,1px 100%!important;background-position:center!important;background-repeat:no-repeat!important;}</style>"""))
if 'go2_k21_play' not in globals():raise RuntimeError('Notebook21 player definitions are incomplete.')
PLAY_UI=go2_k21_play(PROJECT,device='cpu',persist=(MODE=='run'))
display(Markdown('**Playing model:** `'+PLAY_UI.session.metadata['run_id']+'` · optimizer step '+str(PLAY_UI.session.metadata.get('optimizer_step','—'))))
